# Prithvi-EO-1.0 multi-temporal crop classification — DIMER E2E segmentation fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/prithvi-crop-classification-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/prithvi-crop-classification-pipeline/blob/main/tutorials/prithvi_crop_classification_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-ibm--nasa--geospatial%2FPrithvi--EO--1.0--100M--multi--temporal--crop--classification-ffcc4d?style=flat)](https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-1.0-100M-multi-temporal-crop-classification) [![Upstream](https://img.shields.io/badge/Upstream-NASA--IMPACT%2Fhls--foundation--os-181717?style=flat&logo=github&logoColor=white)](https://github.com/NASA-IMPACT/hls-foundation-os) [![Paper](https://img.shields.io/badge/arXiv-2310.18660-b31b1b.svg)](https://arxiv.org/abs/2310.18660)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** 13-class crop and land-cover segmentation of three-date six-band HLS chips with a Prithvi-EO-1.0 temporal ViT encoder, held-out IoU/accuracy against a majority-class baseline, and bounded fine-tuning of the segmentation head to labelled chips

**This notebook is standalone.** It carries the repository's package (4 modules under `src/prithvi_crop_classification_pipeline/`, at revision `9b25a261e1d1`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `b53a88b8da673800b67c34a98a527b77076e7035` (~1680 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh **GPU** runtime installs the pinned dependencies (torch, tifffile, numpy, safetensors, huggingface-hub — no mmcv, mmsegmentation or timm: the network is carried in this notebook), stages and digest-verifies the pinned Prithvi crop-classification checkpoint (1.68 GB) from the Hub, statically audits the mmsegmentation pickle against an allow-list, converts it once into safetensors with a pinned digest (keeping the 98 inference tensors and dropping the training-only auxiliary head and the optimizer state), rebuilds the network from the carried module and loads it strictly, fetches the digest-pinned dataset tarball (1.18 GB, no credential) and extracts exactly the 120 pinned chip and mask members, validates them and assigns roles by spatial block (36 training, 12 validation, 12 test), classifies the held-out chips with the frozen model and scores them against the majority-class baseline, runs a bounded fine-tuning of the segmentation head, scores the same chips again, writes class maps for two held-out chips, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify prediction parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On a T4 the whole path takes a few minutes of model time after the downloads; the tarball and the adaptation are the slowest steps.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own labelled chips as a zip holding `pairs.csv` (columns `id`, `image`, `label`) beside 18-band 224 × 224 GeoTIFF chips (three dates × six HLS bands — blue, green, red, narrow NIR, SWIR 1, SWIR 2 — date-major, surface reflectance × 10 000) and single-band label rasters (0 = no data, 1..13 = the classes in the order the model uses); at least four chips with at least two classes. Your chips are split by seed into training, validation and test sets and flow through the same contract — validation, frozen baseline, adaptation, held-out evaluation, class maps, artifact export and reload parity. The expected schema, the ceilings and the privacy guidance are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

Prithvi-EO-1.0-100M (Jakubik et al., 2023) is NASA and IBM's first foundation model for Harmonized Landsat Sentinel-2 imagery: a ViT-B masked autoencoder pretrained on three-date stacks of six bands over the contiguous United States. The checkpoint packaged here is the upstream authors' fine-tune for multi-temporal crop classification — the first six encoder blocks, a transposed-convolution neck that folds the three dates into one 16×-upsampled feature map, and an FCN head over 13 classes derived from the USDA Cropland Data Layer — trained with mmsegmentation on 224 × 224 chips of three 2022 growing-season dates.

Three things about this row are handled in the open. **The upstream asset is a pickle** — an mmsegmentation checkpoint holding the state dict, the Adam optimizer state and a `meta` record. Section 3 downloads and digest-verifies it, statically lists every global the pickle would import (a state dict of tensors, plus the three data-only names that rebuild one numpy scalar in `meta`), refuses anything outside that allow-list, unpickles it exactly once through torch's weights-only loader with those three names bound to inert stand-ins, keeps the 98 inference tensors and writes a safetensors file whose digest is pinned in the carried module; the network you run is rebuilt from `modeling.py`, carried in this notebook in plain PyTorch, and loads that file strictly. **The dataset ships as one 1.18 GB tarball**, so Section 4 pins it by size and digest, streams through it once and copies out exactly the 120 pinned members (each pinned again by size and digest, no `extractall`, no paths taken from the archive), and leaves the other 1,422 alone. **The model was selected on these chips**: every chip in the archive belongs to the upstream validation split, on which the published checkpoint's best epoch was chosen, so the bounded adaptation in Section 6 is a demonstration of the contract, selected by validation loss with the frozen model as epoch 0; the point of the contract is the same recipe applied to *your* labelled chips.

**Learning objectives:** install the pinned runtime; inspect the carried network, pipeline, dataset and metrics modules; stage and digest-verify a pickled checkpoint, read its static audit and see it converted into safetensors; extract pinned members from a digest-verified tarball and validate real labelled multispectral time series with an ignore class; read per-class IoU, mean IoU, mean class accuracy and overall accuracy against a majority-class baseline; run a bounded fine-tuning of the segmentation head with the upstream class-weighted loss, explicit hyperparameters and frozen BatchNorm statistics; compare the adapted and frozen models on the same held-out chips; write class maps; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** yield estimation, field delineation, dates other than three, chips other than 224 × 224 (tiling is the caller's), cloud masking, the published benchmark scores, the 12-block Prithvi-EO-1.0 encoder (this fine-tune keeps six blocks), and any claim that a 60-chip sample stands in for an operational evaluation. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported **GPU** runtime (Google Colab T4 or better, or a Jupyter kernel with a CUDA GPU and Python 3.12): the network runs in float16 autocast and the default adaptation needs about 4 GB of GPU memory; on CPU one chip takes several seconds and the adaptation would take an hour. About 4.5 GB of disk is needed for the checkpoint, its conversion and the tarball.
- **Knowledge:** what a multispectral surface-reflectance chip is (bands, dates, digital numbers, no-data), what a pixel-wise class map and an ignore class are, and how per-class IoU and mean IoU are read against a majority baseline.
- **Executable serialization handled explicitly:** the pinned checkpoint is a pickle. It is digest-verified, statically audited against an allow-list (audit digest pinned) and unpickled **once** through torch's weights-only loader — with the three data-only numpy names of its `meta` record bound to inert stand-ins — to produce the safetensors the network is actually loaded from. No Hub-hosted Python module is imported and no mmsegmentation code runs; the network is the carried `modeling.py`.
- **Data contract:** a record is `{{id, image, label}}` — a (3, 6, 224, 224) array of three dates × six HLS bands in digital numbers (reflectance × 10 000; an array in [0, 1] is scaled) or an 18-band date-major GeoTIFF, and a (224, 224) mask with classes 0..12 and −1 for no data (or a GeoTIFF with 0 = no data, 1..13 = class). Validation is structural: nothing checks that the bands are the right six in the right order, that the three dates are growing-season dates, or that the label belongs to the chip.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — field-level records tied to a producer or commercial imagery under licence are exactly that. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path fetches one pinned object — the 1.18 GB `validation_chips.tgz` of the Hugging Face dataset `ibm-nasa-geospatial/multi-temporal-crop-classification` at an immutable revision — over HTTPS, digest-verified before any member is read; the dataset is CC BY 4.0 (NASA IMPACT / IBM).
- **External access:** the Hugging Face Hub only, to fetch the pinned `ibm-nasa-geospatial/Prithvi-EO-1.0-100M-multi-temporal-crop-classification` snapshot (~1680 MB in total) at revision `b53a88b8da67…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `tifffile` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'tifffile==2026.9.15',
    'numpy==2.5.3',
    'safetensors==0.8.0',
    'huggingface-hub==1.32.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'prithvi-crop-classification-pipeline',
    'repository_revision': '9b25a261e1d140fb56d6aa892d9aa7538ac9d858',
    'embedded_module': 'src/prithvi_crop_classification_pipeline/pipeline.py',
    'embedded_modules': ['src/prithvi_crop_classification_pipeline/metrics.py', 'src/prithvi_crop_classification_pipeline/modeling.py', 'src/prithvi_crop_classification_pipeline/pipeline.py', 'src/prithvi_crop_classification_pipeline/samples.py'],
    'module_sha256': '7cf2e6630513ac869e38e462b64abf48bfde7fa305164bb5648a07a143d5deb8',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, tifffile
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'tifffile': tifffile.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/prithvi_crop_classification_pipeline/` @ `9b25a261e1d1`)

The next 4 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/4:** `src/prithvi_crop_classification_pipeline/metrics.py`

In [ ]:
"""Pixel-level multi-class segmentation metrics for the crop-classification tutorial and its constant baseline.

All numbers are pooled over the labelled pixels of the chips scored together (the ignore index excluded); nothing
here estimates dispersion. Mean IoU and mean class accuracy average over the classes that occur in the labels or
the predictions of the scored set (mmseg's `nanmean` convention), so a class absent from a small sample does not
pull the mean to zero.
"""

from __future__ import annotations

from collections.abc import Sequence
from typing import Any


def confusion_matrix(predictions: Sequence[Any], labels: Sequence[Any], *, num_classes: int, ignore_index: int) -> Any:
    """(num_classes, num_classes) int64 matrix, rows = label, columns = prediction, ignored pixels excluded."""
    import numpy as np

    matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
    for pred, label in zip(predictions, labels, strict=True):
        pred = np.asarray(pred).reshape(-1).astype(np.int64)
        label = np.asarray(label).reshape(-1).astype(np.int64)
        if pred.shape != label.shape:
            raise ValueError(f"prediction shape {pred.shape} != label shape {label.shape}")
        valid = label != ignore_index
        if valid.any():
            index = label[valid] * num_classes + pred[valid]
            matrix += np.bincount(index, minlength=num_classes * num_classes).reshape(num_classes, num_classes)
    return matrix


def metrics_from_confusion(matrix: Any, class_names: Sequence[str]) -> dict[str, Any]:
    import numpy as np

    matrix = np.asarray(matrix, dtype=np.int64)
    total = int(matrix.sum())
    tp = np.diag(matrix).astype(np.float64)
    label_count = matrix.sum(axis=1).astype(np.float64)
    pred_count = matrix.sum(axis=0).astype(np.float64)
    union = label_count + pred_count - tp
    present = union > 0
    with np.errstate(divide="ignore", invalid="ignore"):
        iou = np.where(present, tp / np.where(union > 0, union, 1), np.nan)
        recall = np.where(label_count > 0, tp / np.where(label_count > 0, label_count, 1), np.nan)
        precision = np.where(pred_count > 0, tp / np.where(pred_count > 0, pred_count, 1), np.nan)
        # F1 is defined for every class that occurs in the labels or the predictions; a class the model never
        # predicts (precision undefined) or that never occurs (recall undefined) scores 0 on the missing side.
        p0, r0 = np.nan_to_num(precision, nan=0.0), np.nan_to_num(recall, nan=0.0)
        f1 = np.where(present, 2 * p0 * r0 / np.where((p0 + r0) > 0, p0 + r0, 1), np.nan)
    return {
        "pixels": total,
        "accuracy": round(float(tp.sum() / total), 4) if total else None,
        "mean_iou": round(float(np.nanmean(iou)), 4) if present.any() else None,
        "mean_accuracy": round(float(np.nanmean(recall)), 4) if (label_count > 0).any() else None,
        "mean_f1": round(float(np.nanmean(f1)), 4) if present.any() else None,
        "iou": {name: (round(float(v), 4) if np.isfinite(v) else None) for name, v in zip(class_names, iou, strict=True)},
        "recall": {name: (round(float(v), 4) if np.isfinite(v) else None) for name, v in zip(class_names, recall, strict=True)},
        "label_fraction": {
            name: round(float(v / total), 4) if total else None for name, v in zip(class_names, label_count, strict=True)
        },
        "classes_scored": int(present.sum()),
    }


def segmentation_metrics(
    predictions: Sequence[Any], labels: Sequence[Any], *, class_names: Sequence[str], ignore_index: int
) -> dict[str, Any]:
    """Per-class IoU / recall, mean IoU, mean class accuracy, mean F1 and overall pixel accuracy."""
    matrix = confusion_matrix(predictions, labels, num_classes=len(class_names), ignore_index=ignore_index)
    return metrics_from_confusion(matrix, class_names)


def majority_baseline(labels: Sequence[Any], *, class_names: Sequence[str], ignore_index: int) -> dict[str, Any]:
    """The constant predictor that names every pixel with the most frequent class of the scored labels — the best
    any constant map can do on these pixels — scored on the same pixels as the model."""
    import numpy as np

    counts = np.zeros(len(class_names), dtype=np.int64)
    for label in labels:
        label = np.asarray(label).reshape(-1).astype(np.int64)
        valid = label[label != ignore_index]
        counts += np.bincount(valid, minlength=len(class_names))[: len(class_names)]
    majority = int(counts.argmax())
    predictions = [np.full(np.asarray(label).shape, majority, dtype=np.int64) for label in labels]
    report = segmentation_metrics(predictions, labels, class_names=class_names, ignore_index=ignore_index)
    report["majority_class"] = class_names[majority]
    return report

**Module 2/4:** `src/prithvi_crop_classification_pipeline/modeling.py` (carried verbatim; see the note above)

In [ ]:
"""The Prithvi-EO-1.0-100M multi-temporal crop-classification network in plain PyTorch.

Vendored from the upstream training code (`NASA-IMPACT/hls-foundation-os`, `geospatial_fm/geospatial_fm.py` at
commit `3b6d401f3b4527059af0e44bd640225285e1933d`, Apache-2.0) and from the parts of `mmsegmentation` 0.30 it was
trained with (`FCNHead` with one conv, the `ConvModule` conv→BN→ReLU order), rewritten without `mmcv`, `mmseg`,
`timm` or `einops` so that the served checkpoint is rebuilt from a dependency-free module. The parameter and
buffer names reproduce the upstream state dict exactly (`backbone.*`, `neck.*`, `decode_head.*`), which is how
the conversion can load it with `strict=True`; the training-only `auxiliary_head` is not part of the network.

Architecture (from the pinned `multi_temporal_crop_classification_Prithvi_100M.py` config):

* `TemporalViTEncoder` — a 3-D patch embedding (`Conv3d`, tubelet 1 × 16 × 16) over (bands, dates, H, W), a
  fixed 3-D sin/cos positional embedding plus a class token, 6 pre-norm transformer blocks of width 768 with 8
  heads (MLP ratio 4), and a final LayerNorm; the first 6 blocks of the 12-block Prithvi-EO-1.0-100M masked
  autoencoder, fine-tuned end to end.
* `ConvTransformerTokensToEmbeddingNeck` — drops the class token, folds the (dates × 14 × 14) tokens into a
  (768 × 3, 14, 14) map, and upsamples it 16× through four `ConvTranspose2d` layers (two with a LayerNorm + GELU).
* `FCNHead` — one 3 × 3 conv (2304 → 256) with BatchNorm and ReLU, Dropout2d(0.1), and a 1 × 1 conv to the 13
  crop / land-cover classes, at the input resolution.

The whole network takes a (B, 6, 3, 224, 224) standardised stack — six HLS bands × three dates, in the exact
layout the upstream data pipeline produced (see `pipeline._normalise`) — and returns (B, 13, 224, 224) logits.
"""

from __future__ import annotations

import math

import torch
from torch import nn


def _sincos_1d(embed_dim: int, positions: torch.Tensor) -> torch.Tensor:
    """(M,) positions -> (M, embed_dim) sin/cos table (float64, as the upstream numpy code)."""
    omega = torch.arange(embed_dim // 2, dtype=torch.float64) / (embed_dim / 2.0)
    omega = 1.0 / torch.pow(10000.0, omega)
    out = positions.to(torch.float64).reshape(-1, 1) * omega.reshape(1, -1)
    return torch.cat([torch.sin(out), torch.cos(out)], dim=1)


def sincos_pos_embed_3d(embed_dim: int, grid_size: tuple[int, int, int], *, cls_token: bool) -> torch.Tensor:
    """The fixed 3-D positional embedding of the upstream `get_3d_sincos_pos_embed`: 6/16 of the width for the
    column, 6/16 for the row and 4/16 for the date, concatenated per token in (date, row, column) order, with a
    zero row prepended for the class token."""
    if embed_dim % 16:
        raise ValueError("embed_dim must be a multiple of 16")
    t_size, h_size, w_size = grid_size
    w_dim = h_dim = embed_dim // 16 * 6
    t_dim = embed_dim // 16 * 4
    w_pos = _sincos_1d(w_dim, torch.arange(w_size)).repeat(t_size * h_size, 1)
    h_pos = _sincos_1d(h_dim, torch.arange(h_size)).repeat_interleave(w_size, dim=0).repeat(t_size, 1)
    t_pos = _sincos_1d(t_dim, torch.arange(t_size)).repeat_interleave(h_size * w_size, dim=0)
    pos = torch.cat([w_pos, h_pos, t_pos], dim=1)
    if cls_token:
        pos = torch.cat([torch.zeros(1, embed_dim, dtype=pos.dtype), pos], dim=0)
    return pos.to(torch.float32)


class PatchEmbed(nn.Module):
    """Frames of images to patch embeddings: a Conv3d with kernel = stride = (tubelet, patch, patch)."""

    def __init__(
        self,
        img_size: int = 224,
        patch_size: int = 16,
        num_frames: int = 3,
        tubelet_size: int = 1,
        in_chans: int = 6,
        embed_dim: int = 768,
    ) -> None:
        super().__init__()
        self.img_size = (img_size, img_size)
        self.patch_size = (patch_size, patch_size)
        self.grid_size = (num_frames // tubelet_size, img_size // patch_size, img_size // patch_size)
        self.num_patches = self.grid_size[0] * self.grid_size[1] * self.grid_size[2]
        self.proj = nn.Conv3d(
            in_chans,
            embed_dim,
            kernel_size=(tubelet_size, patch_size, patch_size),
            stride=(tubelet_size, patch_size, patch_size),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _b, _c, _t, height, width = x.shape
        if (height, width) != self.img_size:
            raise ValueError(f"input spatial size {(height, width)} does not match the model's {self.img_size}")
        x = self.proj(x)  # (B, D, T', H', W')
        return x.flatten(2).transpose(1, 2)  # (B, T'·H'·W', D), date-major token order


class Attention(nn.Module):
    """Multi-head self-attention with a fused qkv projection (timm's `Attention`, no q/k norms)."""

    def __init__(self, dim: int, num_heads: int) -> None:
        super().__init__()
        if dim % num_heads:
            raise ValueError("dim must be divisible by num_heads")
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim**-0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, tokens, dim = x.shape
        qkv = self.qkv(x).reshape(batch, tokens, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(batch, tokens, dim)
        return self.proj(x)


class Mlp(nn.Module):
    def __init__(self, dim: int, hidden: int) -> None:
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.act(self.fc1(x)))


class Block(nn.Module):
    """Pre-norm transformer block (timm's `Block` with `qkv_bias=True`, no layer scale, no drop path)."""

    def __init__(self, dim: int, num_heads: int, mlp_ratio: float = 4.0) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)  # upstream passes nn.LayerNorm unchanged: eps 1e-5
        self.attn = Attention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = Mlp(dim, int(dim * mlp_ratio))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class TemporalViTEncoder(nn.Module):
    """The fine-tuned Prithvi encoder: patch embedding, fixed 3-D positional embedding, class token, blocks."""

    def __init__(
        self,
        img_size: int = 224,
        patch_size: int = 16,
        num_frames: int = 3,
        tubelet_size: int = 1,
        in_chans: int = 6,
        embed_dim: int = 768,
        depth: int = 6,
        num_heads: int = 8,
        mlp_ratio: float = 4.0,
    ) -> None:
        super().__init__()
        self.embed_dim = embed_dim
        self.num_frames = num_frames
        self.patch_embed = PatchEmbed(img_size, patch_size, num_frames, tubelet_size, in_chans, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(
            sincos_pos_embed_3d(embed_dim, self.patch_embed.grid_size, cls_token=True).unsqueeze(0), requires_grad=False
        )
        self.blocks = nn.ModuleList([Block(embed_dim, num_heads, mlp_ratio) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """(B, C, T, H, W) -> (B, 1 + T·H'·W', D) normalised tokens, class token first."""
        x = self.patch_embed(x)
        x = x + self.pos_embed[:, 1:, :]
        cls = (self.cls_token + self.pos_embed[:, :1, :]).expand(x.shape[0], -1, -1)
        x = torch.cat((cls, x), dim=1)
        for block in self.blocks:
            x = block(x)
        return self.norm(x)


class Norm2d(nn.Module):
    """LayerNorm over the channel axis of an NCHW map."""

    def __init__(self, embed_dim: int) -> None:
        super().__init__()
        self.ln = nn.LayerNorm(embed_dim, eps=1e-6)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.ln(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2).contiguous()


class ConvTransformerTokensToEmbeddingNeck(nn.Module):
    """Tokens -> (B, embed_dim, Hp, Wp) map -> 16× upsampled (B, output_embed_dim, 16·Hp, 16·Wp) map through
    four stride-2 transposed convolutions. `embed_dim` is the token width × the number of dates (2304): the
    (dates, Hp, Wp) tokens are folded into the channel axis by a plain reshape, exactly as upstream."""

    def __init__(self, embed_dim: int = 2304, output_embed_dim: int = 2304, hp: int = 14, wp: int = 14) -> None:
        super().__init__()
        self.embed_dim = embed_dim
        self.output_embed_dim = output_embed_dim
        self.hp, self.wp = hp, wp
        self.h_out, self.w_out = hp * 16, wp * 16
        self.fpn1 = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, output_embed_dim, kernel_size=2, stride=2),
            Norm2d(output_embed_dim),
            nn.GELU(),
            nn.ConvTranspose2d(output_embed_dim, output_embed_dim, kernel_size=2, stride=2),
        )
        self.fpn2 = nn.Sequential(
            nn.ConvTranspose2d(output_embed_dim, output_embed_dim, kernel_size=2, stride=2),
            Norm2d(output_embed_dim),
            nn.GELU(),
            nn.ConvTranspose2d(output_embed_dim, output_embed_dim, kernel_size=2, stride=2),
        )

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        x = tokens[:, 1:, :]  # drop the class token
        x = x.permute(0, 2, 1).reshape(x.shape[0], -1, self.hp, self.wp)
        x = self.fpn2(self.fpn1(x))
        return x.reshape(-1, self.output_embed_dim, self.h_out, self.w_out)


class ConvModule(nn.Module):
    """mmcv's ConvModule in its default order: conv (no bias) -> BatchNorm -> ReLU."""

    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, padding: int) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.activate = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activate(self.bn(self.conv(x)))


class FCNHead(nn.Module):
    """mmseg's FCNHead with `num_convs=1`, `concat_input=False`, `dropout_ratio=0.1`."""

    def __init__(self, in_channels: int = 2304, channels: int = 256, num_classes: int = 13, dropout_ratio: float = 0.1) -> None:
        super().__init__()
        self.convs = nn.Sequential(ConvModule(in_channels, channels, kernel_size=3, padding=1))
        self.dropout = nn.Dropout2d(dropout_ratio)
        self.conv_seg = nn.Conv2d(channels, num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv_seg(self.dropout(self.convs(x)))


class PrithviCropSegmenter(nn.Module):
    """Encoder + neck + head: (B, bands, dates, 224, 224) standardised -> (B, num_classes, 224, 224) logits."""

    def __init__(
        self,
        *,
        img_size: int = 224,
        patch_size: int = 16,
        num_frames: int = 3,
        in_chans: int = 6,
        embed_dim: int = 768,
        depth: int = 6,
        num_heads: int = 8,
        head_channels: int = 256,
        num_classes: int = 13,
    ) -> None:
        super().__init__()
        grid = img_size // patch_size
        self.backbone = TemporalViTEncoder(img_size, patch_size, num_frames, 1, in_chans, embed_dim, depth, num_heads)
        self.neck = ConvTransformerTokensToEmbeddingNeck(embed_dim * num_frames, embed_dim * num_frames, grid, grid)
        self.decode_head = FCNHead(embed_dim * num_frames, head_channels, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.decode_head(self.neck(self.backbone(x)))
        if tuple(logits.shape[-2:]) != tuple(x.shape[-2:]):  # mmseg resizes the head output to the input size
            logits = nn.functional.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return logits


def parameter_count(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters())


def check_shapes(module: PrithviCropSegmenter) -> dict[str, int]:
    """Sanity numbers used by the tests and the conversion: tensors, parameters and buffers."""
    state = module.state_dict()
    return {
        "tensors": len(state),
        "elements": sum(int(math.prod(v.shape)) for v in state.values()),
        "parameters": parameter_count(module),
    }

**Module 3/4:** `src/prithvi_crop_classification_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Prithvi-EO-1.0-100M multi-temporal crop classification
(`ibm-nasa-geospatial/Prithvi-EO-1.0-100M-multi-temporal-crop-classification`) DIMER pipeline: verified snapshot,
one-time conversion of the pickled mmsegmentation checkpoint into safetensors, 13-class crop / land-cover
segmentation of three-date six-band HLS chips, held-out evaluation against a majority-class baseline, and bounded
fine-tuning of the segmentation head to a user's labelled chips with a portable adapter.

Prithvi-EO-1.0-100M (Jakubik et al., 2023) is a ViT-B masked-autoencoder foundation model for Harmonized Landsat
Sentinel-2 imagery, pretrained on three-date stacks. The checkpoint packaged here is the upstream authors'
fine-tune for multi-temporal crop classification: the first six encoder blocks, a transposed-convolution neck and
an FCN head over 13 USDA Cropland Data Layer classes, trained on 224 × 224 chips of three HLS dates (six bands
each) over the contiguous United States in 2022.

The upstream asset is an mmsegmentation checkpoint — a torch zip archive whose pickle holds `meta`, `state_dict`
and the Adam `optimizer` state, and references, besides `collections.OrderedDict`, `torch._utils._rebuild_tensor_v2`
and two storage classes, three data-only globals that rebuild the numpy scalar `meta.hook_msgs.best_score`
(`numpy.core.multiarray.scalar`, `numpy.dtype`, `_codecs.encode`). `audit_pickle` verifies statically that nothing
else is referenced; the conversion then unpickles the file **once** with torch's weights-only unpickler, in which
those three names are bound to inert stand-ins (no numpy code runs and the value is discarded), keeps the 98
inference tensors (the training-only auxiliary head and the optimizer are dropped) after a strict load into the
vendored architecture (`modeling.py`, plain PyTorch, no mmcv / mmseg / timm), and writes safetensors with a
pinned digest — the only file the model is ever loaded from.

Everything model-related is imported lazily so that snapshot verification, the pickle audit and input validation
run (and can refuse) before `torch` is imported (fleet RTM-001). `numpy` and `tifffile` are used for chips and are
imported freely.
"""

from __future__ import annotations

import hashlib
import io
import json
import math
import pickletools
import time
import zipfile
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

MODEL_ID = "ibm-nasa-geospatial/Prithvi-EO-1.0-100M-multi-temporal-crop-classification"
MODEL_REVISION = "b53a88b8da673800b67c34a98a527b77076e7035"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "prithvi-eo-1.0-100m-crop"
ARTIFACT_FORMAT = "org.valcorza.prithvi-crop-classification.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Immutable upstream source asset (an mmsegmentation checkpoint, i.e. a pickle; see docs/WEIGHTS.md).
SOURCE_CKPT_NAME = "multi_temporal_crop_classification_Prithvi_100M.pth"
SOURCE_CKPT_BYTES = 1_680_468_041
SOURCE_CKPT_SHA256 = "37ed41637eccccec65ca2031324e2c03a4f168e1ea0ea71ad180910589fa018c"
# Code-free serving file produced deterministically by `convert_model` (asset spec §11.2).
CONVERTED_WEIGHTS_NAME = "prithvi-eo-1.0-100m-crop.safetensors"
CONVERTED_SHA256 = "d1df8044700a0d1e00b11b1fbac66e0495cf6647e1632a858c003eaf4b5ce36d"
CONVERTED_BYTES = 537_722_508
# Static-audit digest of the source pickle (sorted global names), see `audit_pickle`.
PICKLE_AUDIT_SHA256 = "465513635354b8c8a9c4bfd4a37b39d5306eb8ec63527905ce0195075d04108d"
TORCH_GLOBALS = frozenset(
    {"collections.OrderedDict", "torch._utils._rebuild_tensor_v2", "torch.FloatStorage", "torch.LongStorage"}
)
# The three data-only globals that rebuild the numpy scalar `meta.hook_msgs.best_score` (the best validation
# mIoU mmseg logged). They construct values, not code, and the conversion binds them to inert stand-ins.
META_GLOBALS = frozenset({"numpy.core.multiarray.scalar", "numpy.dtype", "_codecs.encode"})
CKPT_ALLOWED_GLOBALS = TORCH_GLOBALS | META_GLOBALS
TRAINING_ONLY_PREFIX = "auxiliary_head."  # mmseg's auxiliary FCN head: trained with, never used at inference

# Architecture (the pinned multi_temporal_crop_classification_Prithvi_100M.py config) and data-contract facts.
IMAGE_SIZE = 224
PATCH_SIZE = 16
NUM_FRAMES = 3
EMBED_DIM = 768
DEPTH = 6
NUM_HEADS = 8
HEAD_CHANNELS = 256
SOURCE_STATE_TENSORS = 112  # 78 backbone + 12 neck + 8 decode_head + 14 auxiliary_head
STATE_TENSORS = 98
STATE_NUMEL = 134_428_174
PARAMETER_COUNT = 134_427_661  # nn.Parameters (the fixed pos_embed included); the rest are BatchNorm buffers
CLASS_NAMES: tuple[str, ...] = (
    "Natural Vegetation",
    "Forest",
    "Corn",
    "Soybeans",
    "Wetlands",
    "Developed/Barren",
    "Open Water",
    "Winter Wheat",
    "Alfalfa",
    "Fallow/Idle Cropland",
    "Cotton",
    "Sorghum",
    "Other",
)
NUM_CLASSES = len(CLASS_NAMES)
# The upstream training loss weights (CrossEntropyLoss class_weight of the pinned config), reused by `adapt`.
CLASS_WEIGHTS: tuple[float, ...] = (
    0.386375,
    0.661126,
    0.548184,
    0.640482,
    0.876862,
    0.925186,
    3.249462,
    1.542289,
    2.175141,
    2.272419,
    3.062762,
    3.626097,
    1.198702,
)
IGNORE_INDEX = -1
# Label rasters of the dataset use 0 = no data and 1..13 = class; records use 0..12 and IGNORE_INDEX (mmseg's
# `reduce_zero_label`), and `read_mask` / `write_sample_pair` translate between the two.
BANDS: tuple[str, ...] = ("BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2")
# Per-band statistics of the pinned config, in the dataset's digital-number units (HLS reflectance × 10 000),
# repeated for each of the three dates.
MEANS: tuple[float, ...] = (494.905781, 815.239594, 924.335066, 2968.881459, 2634.621962, 1739.579917)
STDS: tuple[float, ...] = (284.925432, 357.84876, 575.566823, 896.601013, 951.900334, 921.407808)
REFLECTANCE_SCALE = 10_000.0  # applied only when a chip arrives as reflectance in [0, 1]
VALUE_RANGE = (-2_000.0, 20_000.0)  # plausible HLS digital numbers (clouds and snow exceed 10 000)
MIN_RECORDS = 4
MAX_RECORDS = 2_000
ADAPTATION_MODES = ("head", "head+last_block")  # the only scopes an adapter may declare
LAST_BLOCK_PREFIX = f"backbone.blocks.{DEPTH - 1}."
TRAINABLE_PREFIXES: dict[str, tuple[str, ...]] = {
    "head": ("decode_head.",),
    "head+last_block": ("decode_head.", LAST_BLOCK_PREFIX),
}


# --------------------------------------------------------------------------------------------------
# manifest, staging, static pickle audit and conversion
# --------------------------------------------------------------------------------------------------


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"no snapshot manifest at {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    listed = {entry["path"] for entry in manifest["files"]}
    if SOURCE_CKPT_NAME not in listed:
        raise ValueError(f"manifest does not list {SOURCE_CKPT_NAME}; refusing to proceed")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256_file(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
        if entry["path"] == SOURCE_CKPT_NAME and (size, digest) != (SOURCE_CKPT_BYTES, SOURCE_CKPT_SHA256):
            raise ValueError(f"{entry['path']}: manifest digest disagrees with the package constant")
    return manifest


def verify_converted(path: str | Path | None = None) -> dict[str, Any]:
    """Check the converted serving file (safetensors) against the pinned digest."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    file_path = root / CONVERTED_WEIGHTS_NAME
    if not file_path.is_file():
        raise FileNotFoundError(f"converted file missing: {file_path}")
    size = file_path.stat().st_size
    if size != CONVERTED_BYTES:
        raise ValueError(f"{CONVERTED_WEIGHTS_NAME}: size {size} != pinned {CONVERTED_BYTES}")
    digest = _sha256_file(file_path)
    if digest != CONVERTED_SHA256:
        raise ValueError(f"{CONVERTED_WEIGHTS_NAME}: sha256 {digest} != pinned {CONVERTED_SHA256}")
    return {"files": [{"path": CONVERTED_WEIGHTS_NAME, "bytes": size, "sha256": digest}]}


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the snapshot against its DIMER manifest (size + SHA-256 of every listed Hub file) and, when the
    converted serving file is present, that against the pinned digest."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _verify_manifest(root, MODEL_ID, MODEL_REVISION)
    converted = (root / CONVERTED_WEIGHTS_NAME).is_file()
    if converted:
        verify_converted(root)
    return {**manifest, "converted": converted}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest entries that are absent locally (a fresh clone commits the manifest and git-ignores the
    1.68 GB checkpoint and the safetensors it converts to)."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _pickle_globals(data: bytes) -> dict[str, int]:
    """Every global a pickle stream would import, collected with `pickletools.genops` (no execution)."""
    found: dict[str, int] = {}
    stack: list[Any] = []
    for op, arg, _pos in pickletools.genops(io.BytesIO(data)):
        if op.name == "GLOBAL":  # pickletools renders the (module, name) pair space-separated
            key = arg.replace("\n", " ").replace(" ", ".", 1)
            found[key] = found.get(key, 0) + 1
        elif op.name == "STACK_GLOBAL":
            key = f"{stack[-2]}.{stack[-1]}"
            found[key] = found.get(key, 0) + 1
        if op.name in ("SHORT_BINUNICODE", "BINUNICODE", "UNICODE", "SHORT_BINSTRING", "BINSTRING"):
            stack.append(arg)
        elif op.name in ("MEMOIZE", "BINPUT", "LONG_BINPUT", "PUT"):
            pass
        else:
            stack.append(None)
    return found


def audit_pickle(path: str | Path, *, allowed: frozenset[str] = CKPT_ALLOWED_GLOBALS) -> dict[str, Any]:
    """Statically list the globals a pickle (plain, or inside a torch zip archive) would import and refuse any
    outside `allowed`. Executes nothing. Returns the sorted globals, which of them are the data-only meta
    globals, and their digest."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"file not found: {file_path}")
    data = file_path.read_bytes()
    found: dict[str, int] = {}
    nested = 0
    if data[:4] == b"PK\x03\x04":
        archive = zipfile.ZipFile(io.BytesIO(data))
        for name in archive.namelist():
            if name.endswith(".pkl"):
                nested += 1
                for key, count in _pickle_globals(archive.read(name)).items():
                    found[key] = found.get(key, 0) + count
    else:
        found = _pickle_globals(data)
    violations = sorted(name for name in found if name not in allowed)
    summary = {
        "file": file_path.name,
        "torch_archive": data[:4] == b"PK\x03\x04",
        "pickles": nested if nested else 1,
        "globals": sorted(found),
        "meta_globals": sorted(name for name in found if name in META_GLOBALS),
        "violations": violations,
        "audit_sha256": hashlib.sha256("\n".join(sorted(found)).encode("utf-8")).hexdigest(),
    }
    if violations:
        raise ValueError(f"{file_path.name}: pickle audit failed, globals outside the allow-list: {violations}")
    return summary


def _check_pinned_source(root: Path) -> dict[str, Any]:
    source = root / SOURCE_CKPT_NAME
    if not source.is_file():
        raise FileNotFoundError(f"source file not found: {source}")
    size = source.stat().st_size
    if size != SOURCE_CKPT_BYTES:
        raise ValueError(f"{SOURCE_CKPT_NAME}: size {size} != pinned {SOURCE_CKPT_BYTES}")
    digest = _sha256_file(source)
    if digest != SOURCE_CKPT_SHA256:
        raise ValueError(f"{SOURCE_CKPT_NAME}: sha256 {digest} != pinned {SOURCE_CKPT_SHA256}")
    audit = audit_pickle(source)
    if audit["audit_sha256"] != PICKLE_AUDIT_SHA256:
        raise ValueError(f"{SOURCE_CKPT_NAME}: pickle audit digest {audit['audit_sha256']} != pinned {PICKLE_AUDIT_SHA256}")
    return {"path": SOURCE_CKPT_NAME, "bytes": size, "sha256": digest, "audit": audit}


class _MetaValue:
    """Inert stand-in for the numpy dtype / scalar objects mmseg stored in the checkpoint's `meta`: the
    weights-only unpickler builds these instead of importing numpy, and the conversion discards them."""

    def __init__(self, kind: str, *args: Any) -> None:
        self.kind = kind
        self.args = args

    def __setstate__(self, state: Any) -> None:
        self.state = state

    def __repr__(self) -> str:
        return f"<meta {self.kind}>"


def _meta_scalar(dtype: Any, payload: Any) -> _MetaValue:
    return _MetaValue("numpy.core.multiarray.scalar", dtype, payload)


def _meta_dtype(*args: Any) -> _MetaValue:
    return _MetaValue("numpy.dtype", *args)


def _meta_encode(text: str, encoding: str = "latin1", *_rest: Any) -> bytes:
    return text.encode(encoding)


def restricted_load(path: str | Path) -> dict[str, Any]:
    """Unpickle the checkpoint once through torch's weights-only unpickler, with the three meta globals bound to
    inert stand-ins (nothing from numpy is imported or executed by the pickle). Only the tensors and plain
    containers survive; the stand-ins are what `meta.hook_msgs.best_score` unpickles to."""
    import torch

    bindings = [
        (_meta_scalar, "numpy.core.multiarray.scalar"),
        (_meta_dtype, "numpy.dtype"),
        (_meta_encode, "_codecs.encode"),
        _MetaValue,
    ]
    source = Path(path)
    with torch.serialization.safe_globals(bindings):
        payload = torch.load(source, map_location="cpu", weights_only=True)
    if not isinstance(payload, dict):
        raise ValueError(f"{source.name} did not unpickle to a checkpoint dict")
    return payload


def build_model() -> Any:
    """Instantiate the fine-tuned architecture from the vendored module (no pretrained download)."""
    pass  # standalone rewrite (build_notebook.py): `from .modeling import PrithviCropSegmenter` removed — names are kernel globals defined by the carried modules

    return PrithviCropSegmenter(
        img_size=IMAGE_SIZE,
        patch_size=PATCH_SIZE,
        num_frames=NUM_FRAMES,
        in_chans=len(BANDS),
        embed_dim=EMBED_DIM,
        depth=DEPTH,
        num_heads=NUM_HEADS,
        head_channels=HEAD_CHANNELS,
        num_classes=NUM_CLASSES,
    )


def convert_model(path: str | Path | None = None) -> dict[str, Any]:
    """Convert the pinned mmsegmentation checkpoint into safetensors, deterministically, after size, digest and
    static-audit checks: the restricted weights-only unpickle, the `state_dict` entry with the training-only
    auxiliary head dropped, a strict load into the vendored architecture, and the model's own state dict saved.
    The optimizer state and `meta` are read and discarded."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    source = _check_pinned_source(root)
    import torch
    from safetensors.torch import save_file

    started = time.perf_counter()
    payload = restricted_load(root / SOURCE_CKPT_NAME)
    if "state_dict" not in payload:
        raise ValueError(f"{SOURCE_CKPT_NAME} did not unpickle to an mmsegmentation checkpoint with a state_dict")
    state = payload["state_dict"]
    if not isinstance(state, dict) or any(not isinstance(v, torch.Tensor) for v in state.values()):
        raise ValueError(f"{SOURCE_CKPT_NAME}: state_dict is not a dict of tensors")
    if len(state) != SOURCE_STATE_TENSORS:
        raise ValueError(f"{SOURCE_CKPT_NAME}: state_dict has {len(state)} tensors, expected {SOURCE_STATE_TENSORS}")
    dropped = sorted(k for k in state if k.startswith(TRAINING_ONLY_PREFIX))
    kept = {k: v for k, v in state.items() if not k.startswith(TRAINING_ONLY_PREFIX)}
    model = build_model()
    model.load_state_dict(kept, strict=True)
    canonical = {k: v.contiguous() for k, v in model.state_dict().items()}
    n_elements = sum(v.numel() for v in canonical.values())
    if len(canonical) != STATE_TENSORS or n_elements != STATE_NUMEL:
        raise ValueError(
            f"converted state dict has {len(canonical)} tensors / {n_elements} elements; expected {STATE_TENSORS} / {STATE_NUMEL}"
        )
    save_file(canonical, str(root / CONVERTED_WEIGHTS_NAME), metadata={"format": "pt"})
    report = verify_converted(root)
    meta = payload.get("meta") if isinstance(payload.get("meta"), dict) else {}
    return {
        "source": {k: v for k, v in source.items() if k != "audit"},
        "audit": source["audit"],
        "checkpoint": {
            "epoch": meta.get("epoch"),
            "iter": meta.get("iter"),
            "mmseg_version": meta.get("mmseg_version"),
            "mmcv_version": meta.get("mmcv_version"),
            "entries": sorted(payload),
            "state_dict_tensors": len(state),
            "dropped_training_only_tensors": len(dropped),
            "optimizer_state_discarded": "optimizer" in payload,
        },
        "converted": report["files"],
        "seconds": round(time.perf_counter() - started, 2),
    }


# --------------------------------------------------------------------------------------------------
# chips, labels and validation (no model import)
# --------------------------------------------------------------------------------------------------

INPUT_SCHEMA: dict[str, Any] = {
    "record": (
        "{id, image, label?}: image = (3, 6, 224, 224) float32 stack of three dates × six HLS bands in digital "
        "numbers (reflectance × 10 000; or a GeoTIFF path with 18 bands, date-major); "
        "label = (224, 224) int mask with 0..12 / -1 (or a GeoTIFF path with 0 = no data, 1..13 = class), optional"
    ),
    "bands": list(BANDS),
    "dates": NUM_FRAMES,
    "image_size": IMAGE_SIZE,
    "value_units": (
        "HLS surface-reflectance digital numbers (reflectance × 10 000, int16 in the dataset) in "
        f"[{VALUE_RANGE[0]:.0f}, {VALUE_RANGE[1]:.0f}]; a chip whose values all lie in [0, 1.5] is taken as reflectance "
        "and scaled by 10 000"
    ),
    "classes": {str(i): name for i, name in enumerate(CLASS_NAMES)},
    "ignore_index": IGNORE_INDEX,
    "label_files": "0 = no data (ignored), 1..13 = the classes above in order (the dataset's CDL reclassification)",
    "records": [MIN_RECORDS, MAX_RECORDS],
    "validation": (
        "record shape, date and band counts, chip size, finiteness, value range and label values only. Nothing "
        "checks that the bands are the six HLS bands in the right order, that the three dates are the growing-"
        "season dates the model was trained on, or that the label was drawn for this chip -- any (3, 6, 224, 224) "
        "array is classified without complaint"
    ),
}


def _read_tiff(path: Path) -> Any:
    """Read a GeoTIFF's pixel array with tifffile as (bands, H, W) or (H, W); no georeferencing is used."""
    import numpy as np
    import tifffile

    with tifffile.TiffFile(path) as tf:
        array = tf.asarray()
    if array.ndim == 3 and array.shape[-1] <= 32 and array.shape[0] > 32:
        array = np.moveaxis(array, -1, 0)  # pixel-interleaved (H, W, bands) -> band-sequential
    return np.asarray(array)


def read_chip(path: str | Path) -> Any:
    """Load a chip from a GeoTIFF as float32 (3, 6, H, W): the file's 18 bands are read date-major (date 1's six
    bands, then date 2's, then date 3's — the dataset's layout)."""
    import numpy as np

    array = _read_tiff(Path(path))
    if array.ndim != 3 or array.shape[0] != NUM_FRAMES * len(BANDS):
        raise ValueError(f"{Path(path).name}: expected an 18-band raster (3 dates × 6 bands), got shape {array.shape}")
    stack = array.reshape(NUM_FRAMES, len(BANDS), *array.shape[1:])
    return np.ascontiguousarray(stack.astype(np.float32))


def read_mask(path: str | Path) -> Any:
    """Load a label raster (0 = no data, 1..13 = class) from a GeoTIFF as int64 (H, W) with classes 0..12 and
    no data as IGNORE_INDEX."""
    import numpy as np

    array = _read_tiff(Path(path))
    if array.ndim == 3:
        if array.shape[0] != 1:
            raise ValueError(f"{Path(path).name}: a label raster must have one band, got shape {array.shape}")
        array = array[0]
    raw = array.astype(np.int64)
    return np.ascontiguousarray(np.where(raw == 0, IGNORE_INDEX, raw - 1))


def _check_record(record: Any, index: int) -> dict[str, Any]:
    import numpy as np

    label_name = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label_name} must be a mapping with id/image[/label]")
    for key in ("id", "image"):
        if key not in record:
            raise ValueError(f"{label_name} is missing {key!r}")
    rid, image = record["id"], record["image"]
    if not isinstance(rid, str) or not rid or len(rid) > 128:
        raise ValueError(f"{label_name}: id must be a non-empty string of at most 128 characters")
    if isinstance(image, str | Path):
        if not Path(image).is_file():
            raise ValueError(f"{label_name}: image file not found: {image}")
        image = read_chip(image)
    try:
        array = np.asarray(image, dtype=np.float32)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{label_name}: image must be a numeric array") from exc
    expected = (NUM_FRAMES, len(BANDS), IMAGE_SIZE, IMAGE_SIZE)
    if array.shape != expected:
        raise ValueError(f"{label_name}: image must have shape {expected}, got {array.shape}")
    if not np.all(np.isfinite(array)):
        raise ValueError(f"{label_name}: image contains non-finite values")
    if float(array.min()) >= 0.0 and float(array.max()) <= 1.5:
        array = array * REFLECTANCE_SCALE
    if float(array.min()) < VALUE_RANGE[0] or float(array.max()) > VALUE_RANGE[1]:
        span = (float(array.min()), float(array.max()))
        raise ValueError(f"{label_name}: digital numbers outside the plausible range {VALUE_RANGE}: {span}")
    item: dict[str, Any] = {"id": rid, "image": np.ascontiguousarray(array.astype(np.float32))}
    label = record.get("label")
    if label is not None:
        if isinstance(label, str | Path):
            if not Path(label).is_file():
                raise ValueError(f"{label_name}: label file not found: {label}")
            label = read_mask(label)
        try:
            mask = np.asarray(label)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{label_name}: label must be an integer array") from exc
        if mask.shape != (IMAGE_SIZE, IMAGE_SIZE):
            raise ValueError(f"{label_name}: label must have shape {(IMAGE_SIZE, IMAGE_SIZE)}, got {mask.shape}")
        if not np.issubdtype(mask.dtype, np.integer) and not np.all(mask == np.round(mask)):
            raise ValueError(f"{label_name}: label values must be integers")
        allowed = set(range(NUM_CLASSES)) | {IGNORE_INDEX}
        found = set(np.unique(mask).astype(int).tolist())
        if not found <= allowed:
            raise ValueError(f"{label_name}: label values {sorted(found - allowed)} outside {sorted(allowed)}")
        item["label"] = np.ascontiguousarray(mask.astype(np.int64))
    for key in ("split", "region", "source", "source_id"):
        if key in record:
            item[key] = record[key]
    return item


def check_record(record: Mapping[str, Any]) -> dict[str, Any]:
    """Validate one record and return its normalised copy (float32 digital numbers, int64 label)."""
    return _check_record(record, 0)


def chip_digest(record: Mapping[str, Any]) -> str:
    checked = _check_record(record, 0)
    digest = hashlib.sha256(checked["image"].tobytes())
    if "label" in checked:
        digest.update(checked["label"].tobytes())
    return digest.hexdigest()


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], chip_digest(r)] for r in records]
    return hashlib.sha256(json.dumps(payload, separators=(",", ":")).encode("utf-8")).hexdigest()


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    require_labels: bool = True,
) -> dict[str, Any]:
    """Structural validation of a chip dataset; raises ValueError before any model import."""
    import numpy as np

    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, label} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    class_pixels = np.zeros(NUM_CLASSES, dtype=np.int64)
    ignored = 0
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        if require_labels and "label" not in item:
            raise ValueError(f"records[{index}] has no label; every record of a labelled dataset needs one")
        if "label" in item:
            valid = item["label"][item["label"] != IGNORE_INDEX]
            class_pixels += np.bincount(valid, minlength=NUM_CLASSES)[:NUM_CLASSES]
            ignored += int(item["label"].size - valid.size)
        checked.append(item)
    labelled = sum("label" in r for r in checked)
    if require_labels and labelled and int((class_pixels > 0).sum()) < 2:
        raise ValueError("fewer than two classes are labelled in the dataset; nothing to learn or evaluate")
    total = int(class_pixels.sum())
    return {
        "records": checked,
        "n_records": len(checked),
        "n_labelled": labelled,
        "image_size": IMAGE_SIZE,
        "dates": NUM_FRAMES,
        "bands": list(BANDS),
        "class_pixel_fraction": {
            CLASS_NAMES[c]: round(float(class_pixels[c]) / total, 4) if total else None for c in range(NUM_CLASSES)
        },
        "classes_present": int((class_pixels > 0).sum()),
        "ignored_pixels": ignored,
        "value_range": [
            round(float(min(r["image"].min() for r in checked)), 1),
            round(float(max(r["image"].max() for r in checked)), 1),
        ],
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def validate_inputs(record: Mapping[str, Any]) -> dict[str, Any]:
    """Validate one record; returns its id, shape, value range and label class fractions."""
    item = _check_record(record, 0)
    report = {
        "id": item["id"],
        "shape": tuple(item["image"].shape),
        "value_range": [round(float(item["image"].min()), 1), round(float(item["image"].max()), 1)],
        "has_label": "label" in item,
    }
    if "label" in item:
        label = item["label"]
        valid = int((label != IGNORE_INDEX).sum())
        report["label_fraction"] = {
            CLASS_NAMES[c]: round(float((label == c).sum()) / max(valid, 1), 4) for c in range(NUM_CLASSES)
        }
        report["ignored_pixels"] = int((label == IGNORE_INDEX).sum())
    return report


# --------------------------------------------------------------------------------------------------
# pipeline
# --------------------------------------------------------------------------------------------------


def _normalise(images: Any) -> Any:
    """(B, 3, 6, H, W) digital numbers -> the (B, 6, 3, H, W) standardised tensor the network was trained on.

    The upstream data pipeline standardised the 18 date-major channels with the per-band statistics repeated
    three times, then reshaped the (18, H, W) array to (6, 3, H, W) with a plain `reshape` — which splits the 18
    channels into six groups of three consecutive channels, not into bands and dates. The checkpoint learned
    that layout (it scores 62 % pixel accuracy this way and 17 % with a bands-by-dates layout on the tutorial
    chips), so this function reproduces the reshape exactly rather than the layout the axis names suggest."""
    import numpy as np

    mean = np.asarray(MEANS * NUM_FRAMES, dtype=np.float32)[None, :, None, None]
    std = np.asarray(STDS * NUM_FRAMES, dtype=np.float32)[None, :, None, None]
    batch = np.asarray(images, dtype=np.float32)
    flat = batch.reshape(batch.shape[0], NUM_FRAMES * len(BANDS), *batch.shape[-2:])  # date-major channels
    flat = (flat - mean) / std
    return np.ascontiguousarray(flat.reshape(batch.shape[0], len(BANDS), NUM_FRAMES, *batch.shape[-2:]))


@dataclass
class PrithviCropPipeline:
    """Crop / land-cover segmentation and bounded head fine-tuning on top of the verified Prithvi crop model."""

    model: Any
    device: str
    weights_dir: Path
    source: str
    adapter: dict[str, Any] | None = None

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
        report: Callable[[dict[str, Any]], None] | None = None,
    ) -> PrithviCropPipeline:
        """Verify, convert if needed, rebuild from the vendored module and strictly load. With
        `require_source=False` the checkpoint may be absent (the DIMER-hosted case) as long as the converted file
        verifies. `report` receives the audit and conversion records when a conversion happens."""
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if require_source:
            stage_missing_files(root, allow_download=allow_download)
            snapshot = verify_snapshot(root)
            if not snapshot["converted"]:
                conversion = convert_model(root)
                if report is not None:
                    report({"conversion": conversion})
                snapshot = verify_snapshot(root)
            elif report is not None:
                report({"conversion": "converted file already present and digest-verified"})
            source = "converted from the manifest-verified source checkpoint"
        else:
            verify_converted(root)
            source = "converted file, pinned digest (source checkpoint not required)"
        import torch
        from safetensors.torch import load_file

        chosen = device or ("cuda" if torch.cuda.is_available() else "cpu")
        if chosen.startswith("cuda") and not torch.cuda.is_available():
            raise ValueError("device='cuda' requested but CUDA is not available")
        model = build_model()
        state = load_file(str(root / CONVERTED_WEIGHTS_NAME))
        model.load_state_dict(state, strict=True)
        n_params = sum(p.numel() for p in model.parameters())
        if n_params != PARAMETER_COUNT:
            raise ValueError(f"rebuilt model has {n_params} parameters, expected {PARAMETER_COUNT}")
        model.to(torch.device(chosen)).eval()
        for param in model.parameters():
            param.requires_grad_(False)
        return cls(model=model, device=chosen, weights_dir=root, source=source)

    # ---- forward ---------------------------------------------------------------------------------------

    def _logits(self, images: Any, *, grad: bool = False) -> Any:
        """(B, 3, 6, H, W) float32 digital numbers -> (B, 13, H, W) float32 logits at input resolution."""
        import torch

        batch = torch.from_numpy(_normalise(images)).to(self.device)
        use_amp = self.device.startswith("cuda")
        context = torch.enable_grad() if grad else torch.inference_mode()
        with context, torch.autocast(device_type=self.device.split(":")[0], dtype=torch.float16, enabled=use_amp):
            logits = self.model(batch)
        return logits.float()

    # ---- inference -------------------------------------------------------------------------------------

    def predict(self, records: Sequence[Mapping[str, Any]], *, batch_size: int = 4) -> dict[str, Any]:
        """Classify every pixel: per record the argmax map (H, W) uint8 with classes 0..12, the softmax scores
        (13, H, W) float32 and the fraction of pixels in each class. Softmax scores are the model's own outputs,
        not calibrated probabilities."""
        import numpy as np
        import torch

        checked = validate_dataset(records, min_records=1, require_labels=False)["records"]
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
            raise ValueError("batch_size must be an int in 1..32")
        started = time.perf_counter()
        predictions = []
        for start in range(0, len(checked), batch_size):
            batch = checked[start : start + batch_size]
            logits = self._logits(np.stack([r["image"] for r in batch]))
            scores = torch.softmax(logits, dim=1).cpu().numpy()
            masks = scores.argmax(axis=1).astype(np.uint8)
            for record, score, mask in zip(batch, scores, masks, strict=True):
                predictions.append(
                    {
                        "id": record["id"],
                        "mask": mask,
                        "scores": score.astype(np.float32),
                        "class_fraction": {CLASS_NAMES[c]: round(float((mask == c).mean()), 4) for c in range(NUM_CLASSES)},
                    }
                )
        return {
            "model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY, "adapted": self.adapter is not None},
            "classes": list(CLASS_NAMES),
            "decision_rule": "argmax over the 13 class scores (no threshold)",
            "predictions": predictions,
            "seconds": round(time.perf_counter() - started, 3),
        }

    def evaluate(self, records: Sequence[Mapping[str, Any]], *, batch_size: int = 4) -> dict[str, Any]:
        """Pixel-level metrics on labelled chips (ignore index excluded): per-class IoU and recall, mean IoU,
        mean class accuracy, mean F1 and overall accuracy, with the majority-class baseline scored on the same
        pixels."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import majority_baseline, segmentation_metrics` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1)["records"]
        started = time.perf_counter()
        result = self.predict(checked, batch_size=batch_size)
        masks = [p["mask"] for p in result["predictions"]]
        labels = [r["label"] for r in checked]
        metrics = segmentation_metrics(masks, labels, class_names=CLASS_NAMES, ignore_index=IGNORE_INDEX)
        return {
            "n_records": len(checked),
            "metric": "pixel IoU / accuracy over the labelled pixels of the held-out chips (ignore index excluded)",
            "model": metrics,
            "baseline_majority": majority_baseline(labels, class_names=CLASS_NAMES, ignore_index=IGNORE_INDEX),
            "adapted": self.adapter is not None,
            "seconds": round(time.perf_counter() - started, 3),
        }

    # ---- adaptation ------------------------------------------------------------------------------------

    def _trainable(self, mode: str) -> list[str]:
        if mode not in ADAPTATION_MODES:
            raise ValueError(f"trainable must be one of {ADAPTATION_MODES}")
        prefixes = TRAINABLE_PREFIXES[mode]
        return sorted(name for name, _param in self.model.named_parameters() if name.startswith(prefixes))

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 6,
        lr: float = 1e-4,
        batch_size: int = 4,
        trainable: str = "head",
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the segmentation head (`trainable="head"`; `"head+last_block"` also unfreezes the
        last encoder block) on labelled chips: the upstream class-weighted cross-entropy over the labelled pixels
        (ignore index excluded), AdamW at a fixed learning rate, seeded horizontal/vertical flips, float16
        autocast with loss scaling on CUDA. Epoch 0 records the frozen model; the epoch with the lowest validation
        loss is kept."""
        if not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not (0.0 < lr <= 1e-2):
            raise ValueError("lr must be in (0, 1e-2]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 16:
            raise ValueError("batch_size must be an int in 1..16")
        names = self._trainable(trainable)
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None
        import numpy as np
        import torch

        torch.manual_seed(seed)
        started = time.perf_counter()
        model = self.model
        name_set = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in name_set)
        params = [p for n, p in model.named_parameters() if n in name_set]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
        use_amp = self.device.startswith("cuda")
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        rng = np.random.default_rng(seed)
        weight = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32, device=self.device)

        def loss_fn(logits: Any, target: Any) -> Any:
            return torch.nn.functional.cross_entropy(logits, target, weight=weight, ignore_index=IGNORE_INDEX)

        def val_loss() -> float | None:
            if val_checked is None:
                return None
            model.eval()
            losses = []
            for start in range(0, len(val_checked), batch_size):
                batch = val_checked[start : start + batch_size]
                logits = self._logits(np.stack([r["image"] for r in batch]))
                target = torch.from_numpy(np.stack([r["label"] for r in batch])).to(self.device)
                losses.append(float(loss_fn(logits, target)))
            return sum(losses) / len(losses)

        initial_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
        try:
            history: list[dict[str, Any]] = []
            entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val_loss": val_loss(), "note": "frozen model"}
            if val_checked is not None:
                entry["val"] = self.evaluate(val_checked, batch_size=batch_size)["model"]
            history.append(entry)
            best_val = entry["val_loss"] if entry["val_loss"] is not None else math.inf
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
            best_epoch = 0
            if progress:
                progress(entry)
            n_steps = 0
            for epoch in range(1, epochs + 1):
                model.train()
                for module in model.modules():  # BatchNorm statistics stay frozen: tiny batches would corrupt them
                    if isinstance(module, torch.nn.modules.batchnorm._BatchNorm):
                        module.eval()
                order = rng.permutation(len(train_checked)).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    batch = [train_checked[i] for i in order[start : start + batch_size]]
                    images = np.stack([r["image"] for r in batch])
                    labels = np.stack([r["label"] for r in batch])
                    if rng.random() < 0.5:
                        images, labels = images[..., ::-1], labels[..., ::-1]
                    if rng.random() < 0.5:
                        images, labels = images[..., ::-1, :], labels[..., ::-1, :]
                    logits = self._logits(np.ascontiguousarray(images), grad=True)
                    target = torch.from_numpy(np.ascontiguousarray(labels)).to(self.device)
                    loss = loss_fn(logits, target)
                    optimiser.zero_grad(set_to_none=True)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimiser)
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    scaler.step(optimiser)
                    scaler.update()
                    losses.append(float(loss.detach()))
                    n_steps += 1
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val_loss": val_loss()}
                if val_checked is not None:
                    entry["val"] = self.evaluate(val_checked, batch_size=batch_size)["model"]
                history.append(entry)
                if progress:
                    progress(entry)
                if entry["val_loss"] is None or entry["val_loss"] < best_val:
                    best_val = entry["val_loss"] if entry["val_loss"] is not None else best_val
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the model as it
            # was before adapt() (trained tensors restored), frozen, with no adapter attached.
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable": trainable,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "lr": lr,
            "batch_size": batch_size,
            "loss": "cross-entropy with the upstream class weights, ignore index excluded",
            "augmentation": "seeded horizontal/vertical flips",
            "batchnorm": "running statistics frozen (eval mode) during adaptation",
            "precision": "float16 autocast + GradScaler" if use_amp else "float32",
            "n_train_records": len(train_checked),
            "n_steps": n_steps,
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts -------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted tensors as safetensors with a manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in self.model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY, "converted_sha256": CONVERTED_SHA256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": _sha256_file(weights_path)}
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return out

    @staticmethod
    def check_artifact_manifest(root: Path, manifest: Mapping[str, Any]) -> tuple[Path, str]:
        """Static checks on an adapter manifest, before any model or weights work: format and version, the pinned
        base and converted digest, exactly one weights entry named `adapter.safetensors` inside the artifact
        directory, and an adaptation mode that is one of the declared scopes. Returns the weights path and mode."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not supported "
                f"(expected {ARTIFACT_FORMAT_VERSION!r})"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError("artifact was adapted from a different base model or revision")
        if base.get("converted_sha256") != CONVERTED_SHA256:
            raise ValueError("artifact records a different converted-base digest")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one weights file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact weights file must be named {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weights file must sit inside the artifact directory")
        adapter = manifest.get("adapter")
        mode = adapter.get("trainable") if isinstance(adapter, Mapping) else None
        if mode not in ADAPTATION_MODES:
            raise ValueError(f"artifact adapter.trainable must be one of {ADAPTATION_MODES}")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path, mode

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, scope and digest, then overwrite exactly the tensors the scope allows."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path, mode = self.check_artifact_manifest(root, manifest)
        expected = self._trainable(mode)
        if sorted(manifest["tensors"]) != expected:
            raise ValueError(
                f"artifact tensor list does not match the {len(expected)} tensors that trainable={mode!r} may change"
            )
        entry = manifest["files"][0]
        if _sha256_file(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from the validated manifest")
        state = self.model.state_dict()
        for key, value in tensors.items():
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(f"artifact tensor {key} has shape {tuple(value.shape)}, model has {tuple(state[key].shape)}")
        merged = dict(state)
        merged.update({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()})
        self.model.load_state_dict(merged, strict=True)
        self.model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": manifest["tensors"], "history": manifest.get("history", [])}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
    ) -> PrithviCropPipeline:
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        cls.check_artifact_manifest(root, manifest)
        pipeline = cls.from_pretrained(
            device=device, weights_dir=weights_dir, allow_download=allow_download, require_source=require_source
        )
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 4/4:** `src/prithvi_crop_classification_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled-chip dataset contract for adapting the crop-classification model: the pinned multi-temporal crop
sample, role assignment by spatial block, BYOD loaders and sample export.

The default dataset is **real**: 60 labelled 224 × 224 chips of the HLS multi-temporal crop classification dataset
(NASA IMPACT / IBM, CC BY 4.0) — three HLS dates of six bands over the contiguous United States in 2022, with a
13-class label derived from the USDA Cropland Data Layer — drawn on 2026-09-20 from the 771 chips of the
`validation_chips.tgz` archive (each an image with its mask). Every one of the 60 chips was part of the upstream
*validation* split, i.e. the split the published checkpoint was selected on (`best_mIoU_epoch_80`), so the frozen
model has seen these chips as validation data but was never trained on them. Roles are assigned per 4 × 4-chip
block of the chip grid (a seeded hash of the block; 36 train / 12 validation / 12 test, each stratified by dominant
class so all 13 classes occur in every role) — chips of one block never straddle roles, but neighbouring blocks
may, so the split is by block, not by region; real data must be split by region. The dataset is distributed as one
1.18 GB gzipped tarball on the Hugging Face Hub; the tarball is pinned by byte size and SHA-256, each pinned member
is pinned again by size and SHA-256 and extracted **without** `extractall` into the cache, and everything else in
the archive (including its macOS `._` resource-fork twins) is left alone. The repository redistributes none of
the chips.

A record is ``{id, image, label}``: a (3, 6, 224, 224) digital-number array (or an 18-band GeoTIFF path) and a
(224, 224) mask with classes 0..12 and -1 = no data (or a GeoTIFF path with 0 = no data, 1..13 = class).
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import tarfile
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import (` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "HLS multi-temporal crop classification chips (NASA IMPACT / IBM)"
CORPUS_RELEASE = (
    "Hugging Face dataset ibm-nasa-geospatial/multi-temporal-crop-classification, 60 chips selected 2026-09-20 from "
    "the validation archive"
)
CORPUS_LICENSE = "CC BY 4.0 (NASA IMPACT / IBM)"
DATASET_ID = "ibm-nasa-geospatial/multi-temporal-crop-classification"
DATASET_REVISION = "f285bb27c8f623a0fb6a44a6fd953c3ad34007d6"
CORPUS_BASE_URL = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/"
TAR_NAME = "validation_chips.tgz"
TAR_BYTES = 1_179_542_384
TAR_SHA256 = "d6e616cc008858a1935a8937e0cdf852d574754273b0655fb696bd29aebd2fd3"
CORPUS_BYTES = 111_617_880  # the 120 pinned members, uncompressed
DEFAULT_CACHE_DIR = Path("weights") / "multi-temporal-crop"
ROLES = ("train", "validation", "test")
BLOCK_SIZE = 4  # chips per side of the grid blocks that roles are assigned by
# (chip key, role by block, image member, image bytes, image sha256, mask member, mask bytes, mask sha256) —
# member paths are relative to the tarball root
SAMPLE_RECORDS: tuple[tuple[str, str, str, int, str, str, int, str], ...] = (
    (
        "chip_065_352",
        "train",
        "validation_chips/chip_065_352_merged.tif",
        1808174,
        "e63542cdc0bcf0fc2c5ae4befc7c0613cf823ca131b7ec23754847b1636a6b4d",
        "validation_chips/chip_065_352.mask.tif",
        52124,
        "fb02241466945389394aeb5f33883e06c155d0bab77f6548dcb0e1f08dce0eba",
    ),
    (
        "chip_084_363",
        "train",
        "validation_chips/chip_084_363_merged.tif",
        1808174,
        "f857f6a8e5bcb358e773539fd7bf800a49c622cfb94cacebf970be2aa95e63ae",
        "validation_chips/chip_084_363.mask.tif",
        52124,
        "e39142ceaa5a05d8b084fd15576e3808c499f4f0bcd8cfcba83ee0b7164133a9",
    ),
    (
        "chip_094_337",
        "train",
        "validation_chips/chip_094_337_merged.tif",
        1808174,
        "ce9d684437c40b9f9fa964f99d598216aff1e3dc8bb26e1ada60f22835d30eec",
        "validation_chips/chip_094_337.mask.tif",
        52124,
        "7ff0e18a1625b91365e6f7f60967334da1b95bbd89c8304e9ddc5cf22ece5a3a",
    ),
    (
        "chip_096_341",
        "train",
        "validation_chips/chip_096_341_merged.tif",
        1808174,
        "3fea1ffe7d51fd1484a5b8004f54c284f2bd0b2fbadce7c8103496e406e0e3f6",
        "validation_chips/chip_096_341.mask.tif",
        52124,
        "34693ae6da7ae187ea747e4f0fc2a1be18f2761ce8207b8cb970296eab2fffd9",
    ),
    (
        "chip_106_441",
        "train",
        "validation_chips/chip_106_441_merged.tif",
        1808174,
        "059726da4c85e6e828d4cba6ac1c846411023200f376e002af5b680db8b0e01e",
        "validation_chips/chip_106_441.mask.tif",
        52124,
        "09a996cca65c15ecf96242505282609f94bd1501e2a401373cb77a06c5da8398",
    ),
    (
        "chip_120_555",
        "train",
        "validation_chips/chip_120_555_merged.tif",
        1808174,
        "b70f53aad07e1a9b7864059e7220d3bcdc30d74ea9f71a0a9d79af49d8a79ba6",
        "validation_chips/chip_120_555.mask.tif",
        52124,
        "f5041034c3bb9b720b96ec73db30ad3a0a06f0e3ba3207521fa4d1048fb82d70",
    ),
    (
        "chip_128_312",
        "train",
        "validation_chips/chip_128_312_merged.tif",
        1808174,
        "68b22054f2509b182466b0b15f338cac442c8c34d17f5a1eee6accf21ecd7ee0",
        "validation_chips/chip_128_312.mask.tif",
        52124,
        "4723fe0c1913b0357a101fdc1655f71076d2df4c6b9d3fcad9a443a464dad7af",
    ),
    (
        "chip_128_482",
        "train",
        "validation_chips/chip_128_482_merged.tif",
        1808174,
        "9c3f6ad7b8efdca2dd89c685d038eb7c6daaa8bba1f60aec29a50e99315217b7",
        "validation_chips/chip_128_482.mask.tif",
        52124,
        "fae18d2a7489af82bb1e4de5412592cf202b5a3592a1ffeefa24aa51106df225",
    ),
    (
        "chip_135_484",
        "train",
        "validation_chips/chip_135_484_merged.tif",
        1808174,
        "9f06e3c8409006319f6bec2671d863355ea8e7179ed839772d8872b557aa0fe4",
        "validation_chips/chip_135_484.mask.tif",
        52124,
        "d8f1ebd287fe29504b687ff1de49c8fcf38dd950887a366942326d9e8e698e2e",
    ),
    (
        "chip_144_487",
        "train",
        "validation_chips/chip_144_487_merged.tif",
        1808174,
        "ae230d8ecb1e198876d481ef3588377040528458e98ddfb41ecd6d74dc1e4a7b",
        "validation_chips/chip_144_487.mask.tif",
        52124,
        "8158fa57455349a1babae69f939399cb957bddaf14ff19036aa3c12668de94c2",
    ),
    (
        "chip_165_447",
        "train",
        "validation_chips/chip_165_447_merged.tif",
        1808174,
        "5423cebb6024d283773f3b6c664529b5b30b7a3fb64af1581d190ca5a97846ae",
        "validation_chips/chip_165_447.mask.tif",
        52124,
        "7ad45688714d11d0e774dfaaf4a196e56eecefe92075c2e8f4670284fcdc1026",
    ),
    (
        "chip_168_274",
        "train",
        "validation_chips/chip_168_274_merged.tif",
        1808174,
        "9deaaf3dd88ca854b6419257d7d728352b64a1361a20d8139586812bf3ad2835",
        "validation_chips/chip_168_274.mask.tif",
        52124,
        "2fb56932d9b3111d23bee6d7327b1c59708e9731e445b2657d3261ee4f880633",
    ),
    (
        "chip_173_350",
        "train",
        "validation_chips/chip_173_350_merged.tif",
        1808174,
        "576eb70cd3987677d96bc2d859022868a7bdbd32879a7589d0a947f719f16085",
        "validation_chips/chip_173_350.mask.tif",
        52124,
        "9b5f98df54cbecac6dc0a0b6df7ae7f2f6224b87c49e7d89bb9659179ee53076",
    ),
    (
        "chip_178_022",
        "train",
        "validation_chips/chip_178_022_merged.tif",
        1808174,
        "c5bd551f640d1abf8f6285ea190f3657132c74189a9f5c37a83d7c813d67df6b",
        "validation_chips/chip_178_022.mask.tif",
        52124,
        "1d2ae12bc1bebe9e527607dd1df74dfd710a011fc7069655a6c359bfd467f986",
    ),
    (
        "chip_184_356",
        "train",
        "validation_chips/chip_184_356_merged.tif",
        1808174,
        "9cc45364c0f7613f0ede41e1bee6921ad6cb9e9bc1feb9eece99e785d8aa9bd3",
        "validation_chips/chip_184_356.mask.tif",
        52124,
        "bf080397034272153049e2eb57dc2b3b9c86948fcd370321e0409a7de93c0109",
    ),
    (
        "chip_199_425",
        "train",
        "validation_chips/chip_199_425_merged.tif",
        1808174,
        "33fff6b9e9114439285e59317ff6e932436fc7864acac6ef634eb54361b58fb7",
        "validation_chips/chip_199_425.mask.tif",
        52124,
        "e7c2fc669f8f64e1cfefbd6d8b7853b00aae9bd7c150d932134a41419fb4d25d",
    ),
    (
        "chip_219_323",
        "train",
        "validation_chips/chip_219_323_merged.tif",
        1808174,
        "17262af4c3aba66de068dc5da1466c3a93e7a7ef255e1d1c5fae15f928442331",
        "validation_chips/chip_219_323.mask.tif",
        52124,
        "b2a2810301afd0565a6de77dcc156dd4ccb5d27aea2292959f749c8544ca866f",
    ),
    (
        "chip_220_286",
        "train",
        "validation_chips/chip_220_286_merged.tif",
        1808174,
        "54ac423ddcc4bd6a8abe7cc8bc78e90249651f41e41d7f5399b52998e523c826",
        "validation_chips/chip_220_286.mask.tif",
        52124,
        "7b6af94771bccf9d1ffd8ccdcfcbde542b1d9bf010cdf25062bf8650ee39ca77",
    ),
    (
        "chip_221_442",
        "train",
        "validation_chips/chip_221_442_merged.tif",
        1808174,
        "c9c3772f2f4aabf0c52a3a83b43a9129db92a56d785fbaa6cea54dd317ab2c64",
        "validation_chips/chip_221_442.mask.tif",
        52124,
        "22c17b696b2d83bff852ae4221dfcf93fe4e7b30ac57a39c5fa14407b47414eb",
    ),
    (
        "chip_227_432",
        "train",
        "validation_chips/chip_227_432_merged.tif",
        1808174,
        "e836131999d42acd9e4517a0f36f43ce6c2d0bdc7bbc6bdfc6a5fd38f741339b",
        "validation_chips/chip_227_432.mask.tif",
        52124,
        "ee46977d225def49158ba2fb77e2864025a0766afdeafb7c566633faf25f5b5e",
    ),
    (
        "chip_229_293",
        "train",
        "validation_chips/chip_229_293_merged.tif",
        1808174,
        "ed6154007d0b01694d866525c4930b7bc82da060504c94dd7f0ef9a234283db3",
        "validation_chips/chip_229_293.mask.tif",
        52124,
        "dfc3880a43155fc9dface5e3c5db82b9836829b87e48b6c1b0044ed8443d7195",
    ),
    (
        "chip_231_274",
        "train",
        "validation_chips/chip_231_274_merged.tif",
        1808174,
        "a44f14f44cbb3f5dcc6aac0fd683df56f1b19be03b6e4ca064decb501a572b73",
        "validation_chips/chip_231_274.mask.tif",
        52124,
        "f33da9adee744d682295f869de6d6d56e4924100062cdf4bd984139d077e228b",
    ),
    (
        "chip_231_595",
        "train",
        "validation_chips/chip_231_595_merged.tif",
        1808174,
        "5527a4e81ae8e974308ee9e8e1d3c8a252cc5a3cad329d8434df9f5c217217d2",
        "validation_chips/chip_231_595.mask.tif",
        52124,
        "2c475d72b4de9b79de039a82efedd0635b1d94c70ca880e6170c1400e9a98dbf",
    ),
    (
        "chip_232_279",
        "train",
        "validation_chips/chip_232_279_merged.tif",
        1808174,
        "fd7133455516f1b16af93bfff02eb64fd08e587f2b871db529241f73e977d094",
        "validation_chips/chip_232_279.mask.tif",
        52124,
        "eb91554cc9f02942cf54a5bf3d8c532a344e103dbc34939a5b4bb614c48e785b",
    ),
    (
        "chip_236_335",
        "train",
        "validation_chips/chip_236_335_merged.tif",
        1808174,
        "c3e491c8bdd8330d99a25c3f53f51f081ccff2fc50a942432e33ebb8e2d2148b",
        "validation_chips/chip_236_335.mask.tif",
        52124,
        "62c9a6d795521caf6991a35660a03c1e438d9407bbf313b2b351601029964315",
    ),
    (
        "chip_243_450",
        "train",
        "validation_chips/chip_243_450_merged.tif",
        1808174,
        "aab5c153a372acfabadb5a2fc11d19b0a9d2c1209507bcf0b62e29fd39efcba9",
        "validation_chips/chip_243_450.mask.tif",
        52124,
        "1561dc15054881629378e8be82cb169d16c63f2dc54e8e475dc5e7ca2b9961ff",
    ),
    (
        "chip_247_333",
        "train",
        "validation_chips/chip_247_333_merged.tif",
        1808174,
        "3e5de54fec956202585b3230d6bacb290625976e142b04c6f781aa5cafcaefde",
        "validation_chips/chip_247_333.mask.tif",
        52124,
        "fc589c9f90770c2c37e6f7d8538f34c154c0a6b059e8a8d6ce1548e2b6a2ccbe",
    ),
    (
        "chip_253_285",
        "train",
        "validation_chips/chip_253_285_merged.tif",
        1808174,
        "abd09ec6abe9c37f69a4eebffee548128c143c943140c77258d98fead6e4752a",
        "validation_chips/chip_253_285.mask.tif",
        52124,
        "ec4089fcc6be711c0d6145d368e9db2c118ba7d948393eeddd29aa5646a251be",
    ),
    (
        "chip_254_285",
        "train",
        "validation_chips/chip_254_285_merged.tif",
        1808174,
        "007a63570ce62518f777092c2d1b02761f005d2232852ec6ca3f711190eac148",
        "validation_chips/chip_254_285.mask.tif",
        52124,
        "21f92de44b1d44a13a6b9909fd1c1e423d7053ecd76dcf035c884395d7f74453",
    ),
    (
        "chip_273_471",
        "train",
        "validation_chips/chip_273_471_merged.tif",
        1808174,
        "65c83afb05e71f60e3d400266aac15a79686771099e184c11592d1fb2dc009ce",
        "validation_chips/chip_273_471.mask.tif",
        52124,
        "7d6e3e41b0da5f59ae9f8e54d4ddab0bc7e79b3fda2216dc11d769faebd9bc42",
    ),
    (
        "chip_281_251",
        "train",
        "validation_chips/chip_281_251_merged.tif",
        1808174,
        "87901340e008a063702d0981d2016d40c8da155c8f01f592e037f8bf20e18882",
        "validation_chips/chip_281_251.mask.tif",
        52124,
        "60a0fa571645842958eed436aaa6abfa341a4cfe98807922b8afd9f7b1b71668",
    ),
    (
        "chip_282_404",
        "train",
        "validation_chips/chip_282_404_merged.tif",
        1808174,
        "706ea131325106696cebb5ba2105852a30c1cea47903c47a313e1aac1fddeddb",
        "validation_chips/chip_282_404.mask.tif",
        52124,
        "fc6cfdcabd4f2b501bafbd5507c5185d74c7698dd8a355d2da3246aec3acc27b",
    ),
    (
        "chip_283_264",
        "train",
        "validation_chips/chip_283_264_merged.tif",
        1808174,
        "d5f3f8a399230ad6f16a53706e9ba0d786b2f700dcacc80eaeba4a6ee6110daa",
        "validation_chips/chip_283_264.mask.tif",
        52124,
        "51d0236d693a38d8b4339dd490fa4e0dde78b853912386ce3fce988691c67768",
    ),
    (
        "chip_330_414",
        "train",
        "validation_chips/chip_330_414_merged.tif",
        1808174,
        "915dbf2ac5c8dcc6942b06d29627467879d0394c89d8bbe5696e3c22dfb2e9aa",
        "validation_chips/chip_330_414.mask.tif",
        52124,
        "7e27ab440089e181061146b994f36fe1ce401235df0514c5914115e4e874d0ea",
    ),
    (
        "chip_330_510",
        "train",
        "validation_chips/chip_330_510_merged.tif",
        1808174,
        "7917ecde092fde069ca8625dd7f3b84b3af83ca82db9d04641bb208971e38004",
        "validation_chips/chip_330_510.mask.tif",
        52124,
        "80153c1ab8d6b114480aed92ea1d6adbdc5a284dccfd34a2f3001e29dc32044d",
    ),
    (
        "chip_330_513",
        "train",
        "validation_chips/chip_330_513_merged.tif",
        1808174,
        "f5fb24d8cdadf8df7c3cd9ac917d43eed555d8d9c9b935fd5776470a8d172688",
        "validation_chips/chip_330_513.mask.tif",
        52124,
        "88d5869d50a96ccf983d575591f9b089082df6363676c675557861af49051d9a",
    ),
    (
        "chip_052_043",
        "validation",
        "validation_chips/chip_052_043_merged.tif",
        1808174,
        "b338d72d7fc657aafe0b0c6856db1b22fa76106b1f05c3bc74df259b22c1f40d",
        "validation_chips/chip_052_043.mask.tif",
        52124,
        "f9882ee1f249873fc76f3d2ecc52c4ddb50b5de299b85ceadec53653b248fb7a",
    ),
    (
        "chip_114_438",
        "validation",
        "validation_chips/chip_114_438_merged.tif",
        1808174,
        "6b1efb1c1d5406985babb9296d48206e05824571286354fbfb8c4fd6a5634b79",
        "validation_chips/chip_114_438.mask.tif",
        52124,
        "d9454c30e6896a0918efa2bd91b6d214cb897426413dec6394634a12690c1854",
    ),
    (
        "chip_117_501",
        "validation",
        "validation_chips/chip_117_501_merged.tif",
        1808174,
        "84c284d780feaaaa411f9e8e7fe860ddcfd5f5b90112a0c31a7c1cc452e18a0b",
        "validation_chips/chip_117_501.mask.tif",
        52124,
        "ce315df5853cf94c0fbf4de86148a868a523167fd6a46be7f2e350e2b12e1cdf",
    ),
    (
        "chip_126_303",
        "validation",
        "validation_chips/chip_126_303_merged.tif",
        1808174,
        "1e8984aa6634c2b18a9eefff02fd46b6c0b693b5ecb89daf49f13e4c50b72d38",
        "validation_chips/chip_126_303.mask.tif",
        52124,
        "9e1ecfec5cd0c77697c7a071d810e5f8f322eddd834ff2d4d608a2c5cb4eadb1",
    ),
    (
        "chip_126_506",
        "validation",
        "validation_chips/chip_126_506_merged.tif",
        1808174,
        "10dda5c3901e5ad4c6efad26f56578131f5ab2fdf010f9ea4f1d215d2e7f614a",
        "validation_chips/chip_126_506.mask.tif",
        52124,
        "1f03e9f71e4b60ba353026df22d5000a9ce9e53de4846f5c56dd98fdf37535ac",
    ),
    (
        "chip_156_476",
        "validation",
        "validation_chips/chip_156_476_merged.tif",
        1808174,
        "5e76c735dbcb10676e1b025ba69fef07fc6be97df8efb99438693db160e76b8a",
        "validation_chips/chip_156_476.mask.tif",
        52124,
        "05c45dacc89bfd17d50759a0d92e0c5cf34da0f4239ed55773c0db05b29d28e9",
    ),
    (
        "chip_213_300",
        "validation",
        "validation_chips/chip_213_300_merged.tif",
        1808174,
        "13399cb90a61f0bba5f7aecae0b3195bfc75e911b5f3b812312c32d3a3a423b5",
        "validation_chips/chip_213_300.mask.tif",
        52124,
        "3dadc16d85882be33af1b74461b18ef1a0c6428b02e7de6bbf5a8d5abe25bfcc",
    ),
    (
        "chip_221_307",
        "validation",
        "validation_chips/chip_221_307_merged.tif",
        1808174,
        "63b268cbc42c48398d7b178ccd72b1cca2e73e36edd7d7475ca0d716c655a1a0",
        "validation_chips/chip_221_307.mask.tif",
        52124,
        "07fc0528c184cb75cd47218d5acdbc9a8948c39f24f532674729cd25793e6692",
    ),
    (
        "chip_225_326",
        "validation",
        "validation_chips/chip_225_326_merged.tif",
        1808174,
        "e442c927ffc0978ca02c382b747a5ab494e69eaace0389808a0385fdebbfdb36",
        "validation_chips/chip_225_326.mask.tif",
        52124,
        "0370a05b4d24398c2f51b829376d7991600ddc3b2d2f30451b3282d232d4e5fd",
    ),
    (
        "chip_227_290",
        "validation",
        "validation_chips/chip_227_290_merged.tif",
        1808174,
        "6eec42de00aab30a1047197416b4c78087ec9fc228cb62ee38d3116d61bcfc23",
        "validation_chips/chip_227_290.mask.tif",
        52124,
        "33283b5da5d113030e98ceeed016a5b9007ed3d216515b643dacc2014f2f2633",
    ),
    (
        "chip_261_443",
        "validation",
        "validation_chips/chip_261_443_merged.tif",
        1808174,
        "f2042785219fb8580f41df877ed37a79ab5ab7aadd8695fccc9f0c70afbefc88",
        "validation_chips/chip_261_443.mask.tif",
        52124,
        "0eaa82b38f6c882e2d72d2b275f425a74b98195c5bbc2a6dc99cfcfcadd95331",
    ),
    (
        "chip_334_470",
        "validation",
        "validation_chips/chip_334_470_merged.tif",
        1808174,
        "5cf931f6675cb496269f1c47d8fa6c99fbd30e27f43aa4a458e7f0155a65b698",
        "validation_chips/chip_334_470.mask.tif",
        52124,
        "85f585f88997025c7ac9a9cb7b621b6de819facab41641d6075f9d0e19d59141",
    ),
    (
        "chip_061_092",
        "test",
        "validation_chips/chip_061_092_merged.tif",
        1808174,
        "02aac755376678f7eaee2fa78b463867342899b26c4b41146e3b962232c09175",
        "validation_chips/chip_061_092.mask.tif",
        52124,
        "b46159e87e0f92ab05bc79dae49dce6898512461aa301b791cecc5701c9fe301",
    ),
    (
        "chip_104_104",
        "test",
        "validation_chips/chip_104_104_merged.tif",
        1808174,
        "c1531d1489b1e24bbf0debccd6992c417e0a2d0dbeec6fa10b7fefa93c029581",
        "validation_chips/chip_104_104.mask.tif",
        52124,
        "764c0c49f2297a78b7ead1b667b015aab55d1ec919f67bfb70e9af763d5e1ce3",
    ),
    (
        "chip_131_433",
        "test",
        "validation_chips/chip_131_433_merged.tif",
        1808174,
        "80c8e613b52fef3e330824d6b64e9b567e8cdead79a93018e82b04552b0a2617",
        "validation_chips/chip_131_433.mask.tif",
        52124,
        "90cc7b77573a30fd5f7ef634207816ca8e314992bfc22a9588a144cdc43ee5a8",
    ),
    (
        "chip_138_430",
        "test",
        "validation_chips/chip_138_430_merged.tif",
        1808174,
        "5116cc251db483fdc539c1c8a9dced4139279f4e096f7a9046bda5aa7edee1a2",
        "validation_chips/chip_138_430.mask.tif",
        52124,
        "48717e3d5b1c47d0f6d3e587f79588b11e0ea63f8d94c90bb588c05ad20d3d59",
    ),
    (
        "chip_164_477",
        "test",
        "validation_chips/chip_164_477_merged.tif",
        1808174,
        "cc4613243afab7af6d9a1bc8af5924f82061a3cf51604ec78208d119fe55c730",
        "validation_chips/chip_164_477.mask.tif",
        52124,
        "477555a10370922f92a3ad2a1e62051089f9235795236bc379bfb394e7585ec5",
    ),
    (
        "chip_186_422",
        "test",
        "validation_chips/chip_186_422_merged.tif",
        1808174,
        "2d7417d78c7411b3f06794847604fda79d95fc0c5e1600a628419ddeadf93a3b",
        "validation_chips/chip_186_422.mask.tif",
        52124,
        "e7f31a9071d356da27a0e80a2c8d57c278978fa8f51deac552e2badcf349f345",
    ),
    (
        "chip_200_318",
        "test",
        "validation_chips/chip_200_318_merged.tif",
        1808174,
        "4836763ec196b76bfe7938159b88c83fb0966570f9c1621826d1c51d165ac71a",
        "validation_chips/chip_200_318.mask.tif",
        52124,
        "4e90c257ba3e16945adabe8ae23305ecc96218a754ef28827de8b87ccaa3250c",
    ),
    (
        "chip_252_597",
        "test",
        "validation_chips/chip_252_597_merged.tif",
        1808174,
        "d729a03a9f59bbeb94682086736c91345f30c5c498a71e198f8f5bf73af00c84",
        "validation_chips/chip_252_597.mask.tif",
        52124,
        "fe80a448310064b60fc23882c12f4615f58d717fc0ae83c2166387616e31b54b",
    ),
    (
        "chip_281_252",
        "test",
        "validation_chips/chip_281_252_merged.tif",
        1808174,
        "01ceb90805f66b33373ed0e862b8d087444f663b717ff9682be09d5635443bfe",
        "validation_chips/chip_281_252.mask.tif",
        52124,
        "56016f8cfea95058bdb29f0f0a1900d99f93588534c1fdcb880ba2a862a4ecb4",
    ),
    (
        "chip_292_455",
        "test",
        "validation_chips/chip_292_455_merged.tif",
        1808174,
        "4c30f264f376fa5b1886b20c0c4607b45cee9cd9dda6681293cfe56dfda8a071",
        "validation_chips/chip_292_455.mask.tif",
        52124,
        "5b7b10ba70cbdff203703d3f17f002b0a9afb05b7cd7c547a6c4a4fbd4c5d926",
    ),
    (
        "chip_331_527",
        "test",
        "validation_chips/chip_331_527_merged.tif",
        1808174,
        "4dfdbc11b64f69d94fb03e3f6d0366c99e4dd43baf681fa35db6228d7948fcf5",
        "validation_chips/chip_331_527.mask.tif",
        52124,
        "32a93ad0269eb0a278c723856e049065e00769bfa03882a5946da48907fc5dce",
    ),
    (
        "chip_420_326",
        "test",
        "validation_chips/chip_420_326_merged.tif",
        1808174,
        "50e547176a760fa218155e336277d797c5b7b0a746768f89a38ba710df5672e2",
        "validation_chips/chip_420_326.mask.tif",
        52124,
        "7a1114ccf449bcd5ddaee33750bcf6b2b3f873290fb954fc14b3bff871adf249",
    ),
)

SAMPLE_LABEL_SOURCE = f"{CORPUS_NAME}; {CORPUS_RELEASE}; {CORPUS_LICENSE}"


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 22), b""):
            digest.update(chunk)
    return digest.hexdigest()


def chip_block(key: str, *, block_size: int = BLOCK_SIZE) -> str:
    """The grid block a chip key (`chip_<row>_<col>`) falls in, e.g. `r02c17` for block row 2, block column 17."""
    _prefix, row, col = key.split("_")
    return f"r{int(row) // block_size:02d}c{int(col) // block_size:02d}"


def _pinned_members() -> dict[str, tuple[int, str]]:
    out = {}
    for _name, _role, image, image_bytes, image_sha, label, label_bytes, label_sha in SAMPLE_RECORDS:
        out[image] = (image_bytes, image_sha)
        out[label] = (label_bytes, label_sha)
    return out


def _hub_download_tarball(destination: Path) -> None:
    from huggingface_hub import hf_hub_download

    hf_hub_download(DATASET_ID, TAR_NAME, repo_type="dataset", revision=DATASET_REVISION, local_dir=str(destination.parent))


def fetch_tarball(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> Path:
    """The pinned dataset tarball in the cache, fetched from the Hub at the immutable revision when absent, and
    refused on a size or SHA-256 mismatch (the 1.18 GB file is hashed once per call)."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    local = cache / TAR_NAME
    if not local.is_file() or local.stat().st_size != TAR_BYTES:
        if fetcher is not None:
            local.write_bytes(fetcher(CORPUS_BASE_URL + TAR_NAME))
        else:
            _hub_download_tarball(local)
    size = local.stat().st_size
    digest = _sha256_file(local)
    if size != TAR_BYTES or digest != TAR_SHA256:
        raise ValueError(f"{TAR_NAME}: {size} bytes with sha256 {digest[:16]}…, pinned {TAR_BYTES} / {TAR_SHA256[:16]}…")
    return local


def extract_pinned_members(tar_path: str | Path, *, cache_dir: str | Path | None = None) -> dict[str, bytes]:
    """Stream through the tarball once and copy out exactly the pinned members (no `extractall`, no paths from
    the archive: each is written under its base name in `cache_dir/chips/`), refusing a size or digest mismatch."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    chips = cache / "chips"
    chips.mkdir(parents=True, exist_ok=True)
    wanted = _pinned_members()
    out: dict[str, bytes] = {}
    with tarfile.open(tar_path, "r:gz") as archive:
        for member in archive:
            if member.name not in wanted or not member.isfile():
                continue
            size, sha = wanted[member.name]
            handle = archive.extractfile(member)
            data = handle.read() if handle is not None else b""
            if len(data) != size or _sha256_bytes(data) != sha:
                raise ValueError(
                    f"{member.name}: {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, pinned {size} / {sha[:16]}…"
                )
            (chips / Path(member.name).name).write_bytes(data)
            out[member.name] = data
    missing = sorted(set(wanted) - set(out))
    if missing:
        raise ValueError(f"tarball does not contain {len(missing)} pinned members, e.g. {missing[:3]}")
    return out


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, dict[str, bytes]]:
    """Every pinned chip's image and mask bytes, keyed by chip key: from the extracted cache when every file is
    present with its pinned digest, otherwise from the (verified) tarball."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    chips = cache / "chips"
    wanted = _pinned_members()
    cached: dict[str, bytes] = {}
    for member, (size, sha) in wanted.items():
        local = chips / Path(member).name
        if local.is_file() and local.stat().st_size == size:
            data = local.read_bytes()
            if _sha256_bytes(data) == sha:
                cached[member] = data
    if len(cached) != len(wanted):
        cached = extract_pinned_members(fetch_tarball(cache_dir=cache, fetcher=fetcher), cache_dir=cache)
    out = {}
    for name, _role, image, *_rest in SAMPLE_RECORDS:
        label = _rest[2]
        out[name] = {"image": cached[image], "label": cached[label]}
    return out


def read_corpus(files: Mapping[str, Mapping[str, bytes]]) -> dict[str, list[dict[str, Any]]]:
    """Decode the verified bytes into `{id, image, label}` records grouped by role (train / validation / test)."""
    import tempfile

    splits: dict[str, list[dict[str, Any]]] = {role: [] for role in ROLES}
    for name, role, image_member, *_rest in SAMPLE_RECORDS:
        if name not in files:
            raise ValueError(f"corpus is missing {name}")
        with tempfile.TemporaryDirectory() as tmp:
            image_path = Path(tmp) / "image.tif"
            label_path = Path(tmp) / "label.tif"
            image_path.write_bytes(files[name]["image"])
            label_path.write_bytes(files[name]["label"])
            image = read_chip(image_path)
            label = read_mask(label_path)
        raw = {
            "id": f"{role}-{len(splits[role]):03d}",
            "source_id": name,
            "region": chip_block(name),
            "split": role,
            "image": image,
            "label": label,
            "source": f"{CORPUS_BASE_URL}{TAR_NAME}#{image_member}",
        }
        splits[role].append(check_record(raw))  # range checked, label checked
    return splits


def fetch_sample_dataset(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus (roles by spatial block)."""
    return read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no chip (by pixel digest) and no block (by `region`) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    blocks: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = chip_digest(record)
            if key in seen and seen[key] != name:
                raise ValueError(f"chip {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
            region = record.get("region")
            if region:
                if region in blocks and blocks[region] != name:
                    raise ValueError(f"block {region!r} has chips in both {blocks[region]} and {name}")
                blocks[region] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.25,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train / validation / test after de-duplicating chips. Chips of one
    field or one scene are near-duplicates; group them yourself (one region per split) when that matters."""
    import random

    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = chip_digest(record)
        if key not in seen:
            seen.add(key)
            unique.append(record)
    rng = random.Random(seed)
    rng.shuffle(unique)
    n_test = max(1, round(len(unique) * test_fraction))
    n_val = round(len(unique) * val_fraction)
    splits = {"test": unique[:n_test], "validation": unique[n_test : n_test + n_val], "train": unique[n_test + n_val :]}
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(f"split leaves {len(splits['train'])} training chips; at least {MIN_RECORDS} are required")
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, image, label}` records from a directory or a zip holding `pairs.csv` (columns `id`, `image`,
    `label`) beside 18-band 224 × 224 GeoTIFF chips (three dates × six bands, date-major) and single-band label
    rasters (0 = no data, 1..13 = class); files are decoded from bytes, never extracted to disk."""
    import tempfile

    source = Path(path)
    if source.is_dir():
        table = (source / "pairs.csv").read_text(encoding="utf-8")
        loader = lambda name: (source / name).read_bytes()  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        members = {Path(n).name: n for n in archive.namelist()}
        if "pairs.csv" not in members:
            raise ValueError("BYOD zip must contain pairs.csv")
        table = archive.read(members["pairs.csv"]).decode("utf-8")
        loader = lambda name: archive.read(members[name])  # noqa: E731
    else:
        raise ValueError("BYOD datasets must be a directory or a .zip holding pairs.csv and the GeoTIFF files")
    rows = list(csv.DictReader(io.StringIO(table)))
    missing = {"id", "image", "label"} - set(rows[0].keys() if rows else set())
    if missing:
        raise ValueError(f"pairs.csv is missing columns {sorted(missing)}")
    out = []
    with tempfile.TemporaryDirectory() as tmp:
        for row in rows:
            image_path = Path(tmp) / "image.tif"
            image_path.write_bytes(loader(row["image"]))
            record: dict[str, Any] = {"id": row["id"], "image": read_chip(image_path)}
            if row.get("label"):
                label_path = Path(tmp) / "label.tif"
                label_path.write_bytes(loader(row["label"]))
                record["label"] = read_mask(label_path)
            out.append(record)
    return out


def write_sample_pair(record: Mapping[str, Any], image_path: str | Path, label_path: str | Path) -> dict[str, str]:
    """Write one record as an 18-band int16 TIFF (date-major, digital numbers) and a single-band uint8 TIFF in the
    dataset's label convention (0 = no data, 1..13 = class) — the BYOD shape, without georeferencing — and
    return both paths."""
    import numpy as np
    import tifffile

    image_out, label_out = Path(image_path), Path(label_path)
    image_out.parent.mkdir(parents=True, exist_ok=True)
    image = np.asarray(record["image"], dtype=np.float32).reshape(NUM_FRAMES * len(BANDS), IMAGE_SIZE, IMAGE_SIZE)
    tifffile.imwrite(image_out, np.rint(image).astype(np.int16), photometric="minisblack", planarconfig="separate")
    label = np.asarray(record["label"], dtype=np.int64)
    raw = np.where(label == IGNORE_INDEX, 0, label + 1).astype(np.uint8)
    tifffile.imwrite(label_out, raw, photometric="minisblack")
    return {"image": str(image_out), "label": str(label_out)}


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the pairs table of a split (id, image, label, provenance) in the shape BYOD expects."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "image", "label", "region", "source"])
        writer.writeheader()
        for record in records:
            writer.writerow(
                {
                    "id": record["id"],
                    "image": f"{record.get('source_id', record['id'])}_merged.tif",
                    "label": f"{record.get('source_id', record['id'])}.mask.tif",
                    "region": record.get("region", ""),
                    "source": record.get("source", ""),
                }
            )
    return out


def dataset_manifest(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Validate every split and summarise the dataset (counts, class balance, digests) for provenance exports."""
    summary: dict[str, Any] = {"model_id": MODEL_ID, "image_size": IMAGE_SIZE, "dates": NUM_FRAMES, "splits": {}}
    for name, records in splits.items():
        report = validate_dataset(records, min_records=1)
        summary["splits"][name] = {
            "n_records": report["n_records"],
            "class_pixel_fraction": report["class_pixel_fraction"],
            "classes_present": report["classes_present"],
            "ignored_pixels": report["ignored_pixels"],
            "regions": sorted({str(r.get("region", "")) for r in records if r.get("region")}),
            "digest": report["digest"],
        }
    summary["disjoint"] = check_split_disjoint(splits)
    digests = json.dumps({k: v["digest"] for k, v in summary["splits"].items()}, sort_keys=True)
    summary["digest"] = hashlib.sha256(digests.encode("utf-8")).hexdigest()
    return summary

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `3`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `b53a88b8da67…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `PrithviCropPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), report=print)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "prithvi-eo-1.0-100m-crop",
  "modelId": "ibm-nasa-geospatial/Prithvi-EO-1.0-100M-multi-temporal-crop-classification",
  "revision": "b53a88b8da673800b67c34a98a527b77076e7035",
  "files": [
    {
      "path": "multi_temporal_crop_classification_Prithvi_100M.pth",
      "bytes": 1680468041,
      "sha256": "37ed41637eccccec65ca2031324e2c03a4f168e1ea0ea71ad180910589fa018c"
    },
    {
      "path": "README.md",
      "bytes": 4333,
      "sha256": "eec9be05906903af9a34924233fd4a3284c54ef09c1e77b2a5a74e85b663c726"
    },
    {
      "path": "multi_temporal_crop_classification_Prithvi_100M.py",
      "bytes": 7085,
      "sha256": "ee456e701b6d9fddfe29f35fb649095f65ac39855fdb3ee6df8cbb4e6ca79451"
    }
  ],
  "totalBytes": 1680479459
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = PrithviCropPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), report=print)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Sample chips, validation and roles

The default dataset is 60 labelled 224 × 224 chips of the HLS multi-temporal crop classification dataset — three 2022 HLS dates of six bands each, with a 13-class label derived from the USDA Cropland Data Layer — drawn from the 771 chips of the archive. Roles are assigned per 4 × 4-chip block of the chip grid (36 training, 12 validation, 12 test, each stratified by dominant class so all 13 classes occur in every role): chips of one block never straddle roles, but neighbouring blocks may, so this is a split by block, not by region. `fetch_corpus` downloads the dataset tarball from the Hub at its immutable revision, refuses it on any size or SHA-256 mismatch, streams through it once and copies out exactly the 120 pinned members — each refused on its own size or digest mismatch and written under its base name, never at a path taken from the archive — then reads the 18-band int16 chips as (3, 6, 224, 224) digital numbers and the masks, mapping 0 (no data) to −1 and 1..13 to 0..12. `dataset_manifest` validates every split, checks that no chip or block appears twice and records a digest.

Look for: 36 / 12 / 12 chips with all 13 classes present in each role, the block ids per split, a written sample pair (`outputs/prithvi_crop_classification_sample_chip.tif` + `_sample_label.tif`, the BYOD shape), and three refusal probes — a two-date chip, a mask with an unknown class, a chip with digital numbers far outside range — each rejected before the model runs. The tarball takes about a minute to fetch and two to stream.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np

USE_BYOD = False  # @param {type:"boolean"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    splits = split_dataset(load_byod_dataset(byod_path), seed=0)
    data_source = 'BYOD (' + file_name + ')'
else:
    splits = fetch_sample_dataset(cache_dir='weights/multi-temporal-crop')
    data_source = SAMPLE_LABEL_SOURCE
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']

dataset_report = dataset_manifest({'train': train_records, 'validation': val_records, 'test': test_records})
print({'data_source': data_source, 'splits': {k: v['n_records'] for k, v in dataset_report['splits'].items()}, 'disjoint': dataset_report['disjoint'], 'digest': dataset_report['digest'][:16] + '...'})
for name, part in dataset_report['splits'].items():
    top = sorted(part['class_pixel_fraction'].items(), key=lambda kv: -kv[1])[:4]
    print({name: {'classes_present': part['classes_present'], 'largest_classes': top, 'ignored_pixels': part['ignored_pixels'], 'blocks': len(part['regions'])}})
print({'first_test_chip': validate_inputs(test_records[0])})
sample_pair = write_sample_pair(test_records[0], 'outputs/prithvi_crop_classification_sample_chip.tif', 'outputs/prithvi_crop_classification_sample_label.tif')
print({'sample_pair': sample_pair, 'pairs_csv': str(write_dataset_csv(test_records, 'outputs/prithvi_crop_classification_sample_pairs.csv'))})

print({'validation': INPUT_SCHEMA['validation']})
probes = {
    'two-date chip': [{**test_records[0], 'image': test_records[0]['image'][:2]}, *test_records[1:4]],
    'unknown label class': [{**test_records[0], 'label': np.where(test_records[0]['label'] == 2, 13, test_records[0]['label'])}, *test_records[1:4]],
    'digital numbers out of range': [{**test_records[0], 'image': test_records[0]['image'] * 50.0}, *test_records[1:4]],
}
for name, records in probes.items():
    try:
        validate_dataset(records)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. The frozen model against the majority-class baseline

`pipe.predict` standardises each chip with the band statistics of the upstream training configuration and folds the 18 channels exactly as the upstream data pipeline did (a plain reshape that the checkpoint learned — see the note in `pipeline._normalise`), runs the encoder, neck and head in float16 autocast, and returns the argmax map (classes 0..12), the softmax scores (the model's outputs, not calibrated probabilities) and the class fractions per chip. `pipe.evaluate` pools the labelled pixels of every held-out chip into one 13 × 13 confusion matrix (−1 pixels excluded) and reports the per-class IoU and recall, the mean IoU and mean class accuracy over the classes present, the mean F1 and the overall accuracy; the **majority-class baseline** — every pixel named with the most frequent class of the scored labels, the best any constant map can do — is scored on the same pixels.

Look for: a mean IoU near 0.44 and an accuracy near 0.59 on the test chips (in the build record 0.438 and 0.591, against 0.011 and 0.137 for the majority class, Natural Vegetation; the model card reports a mean IoU of 0.427, an overall accuracy of 60.6 % and a mean class accuracy of 64.1 % on the full validation split), with Open Water and Winter Wheat the easiest classes and Natural Vegetation and Other the hardest. These are sample-sanity numbers on 12 chips, not the benchmark.

In [ ]:
import time

t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records)
frozen_val = pipe.evaluate(val_records)
print({'seconds': round(time.perf_counter() - t0, 1), 'metric': frozen_test['metric']})
print({'baseline_majority_test': {k: frozen_test['baseline_majority'][k] for k in ('majority_class', 'accuracy', 'mean_iou')}})
print({'frozen_test': {k: frozen_test['model'][k] for k in ('mean_iou', 'mean_accuracy', 'mean_f1', 'accuracy', 'classes_scored')}})
print({'frozen_test_iou': frozen_test['model']['iou']})
print({'frozen_validation': {k: frozen_val['model'][k] for k in ('mean_iou', 'accuracy')}})
frozen_predictions = pipe.predict(test_records)
for record, pred in list(zip(test_records, frozen_predictions['predictions']))[:6]:
    labelled = record['label'] >= 0
    dominant = CLASS_NAMES[int(np.bincount(record['label'][labelled], minlength=NUM_CLASSES).argmax())]
    predicted = max(pred['class_fraction'], key=pred['class_fraction'].get)
    print({'chip': record['source_id'], 'block': record['region'], 'dominant_label': dominant, 'dominant_prediction': predicted, 'agreement': round(float((pred['mask'] == record['label'])[labelled].mean()), 3)})
print({'decision_rule': frozen_predictions['decision_rule'], 'scores_shape': frozen_predictions['predictions'][0]['scores'].shape})

## 6. Bounded fine-tuning of the segmentation head

`pipe.adapt` trains the 8 tensors of the FCN head (5.3 M parameters — 4 % of the model) and nothing else: the encoder and the neck are frozen (no gradient is stored for them), and the head's BatchNorm layer keeps its running statistics, because batches of four chips would corrupt them. Each step takes four chips with a seeded horizontal or vertical flip, computes the cross-entropy over the labelled pixels (−1 ignored) with the upstream class weights (rare classes such as Open Water and Sorghum count up to 9× more than Natural Vegetation) and takes an AdamW step at a small fixed learning rate with gradient-norm clipping and float16 loss scaling. Epoch 0 records the frozen model's validation loss and metrics; the epoch with the lowest validation loss is kept — which can be epoch 0, since the packaged checkpoint was selected on the very split these chips come from.

Watch the validation loss: in the build record it fell from 0.937 to 0.911 over four epochs while the validation mean IoU slipped from 0.461 to 0.445 — the class-weighted loss and the mean IoU the checkpoint was selected by do not rank the same head, which is exactly why the kept epoch is chosen on the loss you declare and reported beside the metric you care about. Four epochs (36 steps) take a few minutes on a T4, the validation pass after each epoch included. `TRAINABLE = 'head+last_block'` also unfreezes the last encoder block (7.1 M more parameters).

In [ ]:
EPOCHS = 4  # @param {type:"integer"}
LEARNING_RATE = 1e-5  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}
TRAINABLE = 'head'  # @param ["head", "head+last_block"]

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4), 'val_loss': round(entry['val_loss'], 4)}
    if 'val' in entry:
        row['val_mean_iou'] = entry['val']['mean_iou']
        row['val_accuracy'] = entry['val']['accuracy']
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable=TRAINABLE, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'steps': adapt_result['n_steps'], 'best_epoch': adapt_result['best_epoch'], 'loss': adapt_result['loss'], 'precision': adapt_result['precision'], 'batchnorm': adapt_result['batchnorm'], 'seconds': adapt_seconds})

## 7. Held-out evaluation: the paired comparison

The test chips were never used for training or epoch selection (their blocks were assigned to the test role before anything ran). The adapted model is scored exactly as the frozen model was in Section 5, and the table puts the baseline, the frozen and the adapted numbers side by side. The cell asserts what the procedure guarantees — the kept epoch's validation loss is no higher than the frozen model's, and re-scoring the validation chips reproduces the kept epoch's mean IoU within 0.01 (float16 kernels are not bit-reproducible across batch sizes) — and prints the test numbers without asserting a direction: on this sample the test mean IoU moved from 0.438 to 0.430 and the accuracy from 0.591 to 0.581 in the build record (the kept epoch lowered the class-weighted validation loss, not the mean IoU), a sample-sanity observation on 12 chips with no dispersion estimate, not a quality claim. With your own chips from another region or year, the gap between frozen and adapted is the number to watch.

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
comparison = {}
for key in ('mean_iou', 'mean_accuracy', 'mean_f1', 'accuracy'):
    comparison[key] = {'baseline_majority': frozen_test['baseline_majority'][key], 'frozen': frozen_test['model'][key], 'adapted': adapted_test['model'][key]}
for key, row in comparison.items():
    print({key: row})
moved = {name: (frozen_test['model']['iou'][name], adapted_test['model']['iou'][name]) for name in CLASS_NAMES if frozen_test['model']['iou'][name] != adapted_test['model']['iou'][name]}
print({'per_class_iou_changes': moved if moved else 'none (the kept epoch is the frozen model)'})
print({'validation_mean_iou': {'frozen': frozen_val['model']['mean_iou'], 'adapted': adapted_val['model']['mean_iou']}, 'validation_loss': {'frozen': adapt_result['history'][0]['val_loss'], 'kept_epoch': adapt_result['history'][adapt_result['best_epoch']]['val_loss']}})
evaluation_report = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset': dataset_report,
    'frozen': {'test': frozen_test, 'validation': frozen_val},
    'adapted': {'test': adapted_test, 'validation': adapted_val},
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/prithvi_crop_classification_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
assert adapt_result['history'][adapt_result['best_epoch']]['val_loss'] <= adapt_result['history'][0]['val_loss']
assert abs(adapted_val['model']['mean_iou'] - adapt_result['history'][adapt_result['best_epoch']]['val']['mean_iou']) < 1e-2
print({'report': 'outputs/prithvi_crop_classification_evaluation_report.json'})

## 8. Class maps, artifact export and fresh reload

The adapted model's class maps of two held-out chips are written as single-band GeoTIFFs in the dataset's label convention (0 = no data, 1..13 = class) beside the reference masks, so they can be opened side by side: the agreement per chip printed here is a sanity check, not an evaluation.

`pipe.save_artifact` writes the trained tensors (about 21 MB) as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the converted base file, the adaptation scope, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `PrithviCropPipeline.from_artifact` re-verifies the base file, checks the artifact manifest, scope and digest **before** deserialising, rebuilds the network and overlays the tensors — a fresh object from files, not the in-memory model (VER2). The cell asserts the same held-out mean IoU within 0.001 and score maps within 0.01 (VER4: float16 tolerances; on one device they are usually identical).

In [ ]:
import platform
import shutil

import tifffile

shown_records = test_records[:2]
shown_predictions = pipe.predict(shown_records)
for record, pred in zip(shown_records, shown_predictions['predictions']):
    tifffile.imwrite(f'outputs/prithvi_crop_classification_map_adapted_' + record['source_id'] + '.tif', (pred['mask'].astype(np.int64) + 1).astype(np.uint8))
    tifffile.imwrite(f'outputs/prithvi_crop_classification_map_reference_' + record['source_id'] + '.tif', np.where(record['label'] < 0, 0, record['label'] + 1).astype(np.uint8))
    labelled = record['label'] >= 0
    print({'chip': record['source_id'], 'agreement': round(float((pred['mask'] == record['label'])[labelled].mean()), 3), 'largest_predicted': max(pred['class_fraction'], key=pred['class_fraction'].get), 'note': 'sanity check on two chips'})
with open('outputs/prithvi_crop_classification_predictions.json', 'w', encoding='utf-8') as f:
    json.dump({'model': shown_predictions['model'], 'classes': shown_predictions['classes'], 'decision_rule': shown_predictions['decision_rule'], 'predictions': [{'id': p['id'], 'class_fraction': p['class_fraction']} for p in shown_predictions['predictions']]}, f, indent=2)

artifact_dir = Path('outputs/prithvi_crop_classification_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'prithvi_crop_classification', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'trainable': artifact_manifest['adapter']['trainable'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = PrithviCropPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
reloaded_test = reloaded.evaluate(test_records)
before = pipe.predict(test_records[:2])['predictions']
after = reloaded.predict(test_records[:2])['predictions']
parity = {'mean_iou_diff': round(abs(reloaded_test['model']['mean_iou'] - adapted_test['model']['mean_iou']), 6), 'metrics_identical': reloaded_test['model'] == adapted_test['model'], 'max_abs_score_diff': max(float(np.abs(a['scores'] - b['scores']).max()) for a, b in zip(before, after))}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['mean_iou_diff'] < 1e-3 and parity['max_abs_score_diff'] < 1e-2

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model': {**evaluation_report['model'], 'model_license': MODEL_LICENSE, 'device': pipe.device, 'source': pipe.source},
    'provenance': {
        'source_asset': [e for e in MANIFEST['files'] if e['path'] == SOURCE_CKPT_NAME],
        'pickle_audit_sha256': PICKLE_AUDIT_SHA256,
        'meta_globals_bound_to_stand_ins': sorted(META_GLOBALS),
        'converted': verify_converted(WEIGHTS_DIR)['files'],
        'pickle_unpickled_once_for_conversion': True,
        'served_from_pickle': False,
        'remote_code_executed': False,
        'network_source': 'modeling.py carried in this notebook (plain PyTorch)',
        'data_tarball': {'name': TAR_NAME, 'sha256': TAR_SHA256, 'pinned_members': 2 * len(SAMPLE_RECORDS)},
        'data_base_url': CORPUS_BASE_URL,
        'data_license': CORPUS_LICENSE,
    },
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'tifffile': tifffile.__version__, 'numpy': np.__version__},
    'data_source': data_source,
    'comparison': comparison,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes']},
    'reload_parity': parity,
}
with open('outputs/prithvi_crop_classification_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

## Interpretation and limits

On 12 held-out chips the packaged crop-classification model reaches a mean IoU near 0.44 and an accuracy near 0.59 against a majority-class baseline of 0.01 and 0.14; a bounded fine-tuning of its segmentation head on 36 chips, selected by validation loss with the frozen model as a candidate, lowers the class-weighted validation loss a little and leaves the mean IoU where it was (0.438 → 0.430 on the test chips in the build record). That is the claim: the adaptation contract runs end to end on real labelled multispectral time series drawn from a digest-verified tarball, the pickle is audited and converted rather than served, the network is carried in plain PyTorch, and the artifact that carries the change is 21 MB and reloads with the same outputs. It is not a claim that this sample improves the model — the checkpoint was selected on the split these chips come from — nor that 12 chips measure its skill.

The numbers are sample-sanity evidence: one seeded run, 12 test chips, no dispersion estimate, pixel-pooled metrics that let large fields dominate, and labels derived from the Cropland Data Layer with their own errors at field edges and for minor crops. Nothing here measures the model outside the contiguous United States, outside 2022, on other dates, or on chips larger than 224 × 224.

Three things to carry to real data. **The 18 channels and their scaling are the contract:** three dates × six bands — blue, green, red, narrow NIR, SWIR 1, SWIR 2 — date-major, as digital numbers; and the network folds them exactly as its training pipeline did, which is not the (bands, dates) layout the axis names suggest — a different band order or date order is classified without complaint and silently wrong. **Split by region, not by chip:** neighbouring chips share fields, and a random split makes memorisation look like skill. **Read the baseline and the per-class IoU first:** the majority class alone is right on a seventh of the pixels, and a mean IoU hides that Natural Vegetation is barely found while Open Water is easy.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify a pickled upstream checkpoint, audit and convert it into safetensors without executing anything outside the audited allow-list, rebuild the network from the carried module, fetch a digest-pinned tarball and extract exactly the pinned labelled chips, execute bounded fine-tuning, evaluate against a baseline and the frozen model on held-out chips, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or crop-mapping skill beyond the checks shown.

**Optional experiments (they do not affect the default path):** set `TRAINABLE = 'head+last_block'`; raise `EPOCHS` and watch the validation loss; try `LEARNING_RATE = 1e-4` to see the frozen model win every epoch; or bring your own labelled chips through BYOD and read the baseline before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/prithvi-crop-classification-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/prithvi-crop-classification-pipeline/blob/main/MODEL_CARD.md
- Weights and conversion notes: https://github.com/kurtvalcorza/prithvi-crop-classification-pipeline/blob/main/docs/WEIGHTS.md
- Hugging Face model repository: https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-1.0-100M-multi-temporal-crop-classification (revision `b53a88b8da673800b67c34a98a527b77076e7035`)
- Multi-temporal crop classification dataset: https://huggingface.co/datasets/ibm-nasa-geospatial/multi-temporal-crop-classification (CC BY 4.0)
- Jakubik, J., Roy, S., Phillips, C. E., et al. (2023). Foundation models for generalist geospatial artificial intelligence. arXiv:2310.18660: https://arxiv.org/abs/2310.18660
- Upstream fine-tuning code: https://github.com/NASA-IMPACT/hls-foundation-os (the network is vendored in `modeling.py`)
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)